# Imports

In [1]:
import xpress as xp
import numpy as np 
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

# Model

## Data cleaning and preparation for model

### Extract the collections for degree programmes

In [2]:
programme_df = pd.read_csv('2024-5 DPT Data.csv',encoding = "latin1")
courses_df = pd.read_csv("2024-5 Event Module Room.csv")
departments = [
    'School of Philosophy, Psychology and Language Sciences',
                'School of Economics',
               'School of Informatics',
                'School of Physics and Astronomy',
               'Business School'
]
programmes= ['Mathematics and Business BSc (Hons)',
             'Mathematics and Physics (BSc Hons)',
             'Economics and Mathematics (MA Hons)',
             'Computer Science and Mathematics (BSc Hons)',
              'Philosophy and Mathematics (MA Hons)'
            ]
            
needed = [
     '2024-25: UTMATHB : Y3 options : Business options','2024-25: UTMATHB : Y4/5 : Approved Outside Courses','ROU_H_UT International Business 4_10',
          
           'Economics Course Options for Joint Programmes Year 3 (A)','Topics in Microeconomics', 'Essentials of Econometrics','Economics Course Options Year 3 (joint programmes)','Economics Course Options Year 4 (joint programmes)',
          
          'Electromagnetism and Relativity','Undergraduate (School of Physics and Astronomy) Level 9 and 10  courses','MathsPhysics : Y3 physics choice','MathsPhysics : Y4 Physics Projects','Undergraduate (School of Physics and Astronomy) Level 10 and 11 courses',
          
         'Informatics Hons 3rd Year Group Project and Large Practical','Informatics Hons 3rd Year AI Courses','Informatics Hons 3rd Year Joint Degree CS Courses','Informatics - Professional Issues','Informatics Hons 4th Year Courses',
          
           'Year 3 Philosophy - History of Philosophy','Practical Philosophy', 'Theoretical Philosophy','Philosophy Honours Year 4'
         ]
years = [3,4]
programme_df = programme_df[programme_df['Programme Name'].isin(programmes)]
programme_df = programme_df[programme_df['Programme Year'].isin(years)]
courses_df = courses_df[courses_df['Module Department'].isin(departments)]
programme_df = programme_df[programme_df['Collection Name'].isin(needed)]

In [3]:
#Changes for economics
economics_course_change = (
    (courses_df['Module Name'] == 'Advanced Mathematical Economics, Advanced Mathematical Economics, Advanced Mathematical Economics, Mathematical Microeconomics 1') 
)
courses_df.loc[economics_course_change, 'Module Name'] = 'Advanced Mathematical Economics'

economics_course_change_1 = (
    (courses_df['Module Name'] == 'Advanced Mathematical Economics, Advanced Mathematical Economics, Mathematical Microeconomics 1, Advanced Mathematical Economics') 
)
courses_df.loc[economics_course_change_1, 'Module Name'] = 'Advanced Mathematical Economics'
def filtering(programme_name,module_department,startwith):
    prog = programme_df[programme_df['Programme Name'] == programme_name]
    cour = courses_df[courses_df['Module Department']== module_department]
    all_courses = sorted(cour['Module Name'].unique())
    filtered = prog[~prog['Course Code'].str.startswith(startwith)]
    filtered_courses = sorted(filtered['Course Name'].unique())
    not_there = sorted(list(set(filtered_courses)-set(all_courses)))
    return all_courses, not_there
programme_df['Collection Name'].unique()

array(['Economics Course Options for Joint Programmes Year 3 (A)',
       'Economics Course Options Year 3 (joint programmes)',
       'Economics Course Options Year 4 (joint programmes)',
       'Topics in Microeconomics', 'Essentials of Econometrics',
       'Year 3 Philosophy - History of Philosophy',
       'Practical Philosophy', 'Theoretical Philosophy',
       '2024-25: UTMATHB : Y4/5 : Approved Outside Courses',
       'ROU_H_UT International Business 4_10',
       'Informatics Hons 3rd Year Group Project and Large Practical',
       'Informatics Hons 3rd Year AI Courses',
       'Informatics Hons 3rd Year Joint Degree CS Courses',
       'Informatics - Professional Issues',
       'Informatics Hons 4th Year Courses',
       'MathsPhysics : Y4 Physics Projects',
       'Undergraduate (School of Physics and Astronomy) Level 10 and 11 courses',
       '2024-25: UTMATHB : Y3 options : Business options',
       'Electromagnetism and Relativity',
       'MathsPhysics : Y3 physics ch

In [4]:
a_phy, n_phy = filtering('Mathematics and Physics (BSc Hons)','School of Physics and Astronomy','MAT')
for i in n_phy:
    for j in a_phy:
        l = [x.strip() for x in j.split(',')]
        if i in l:
            condition = ((courses_df['Module Name']==j))
            courses_df.loc[condition,'Module Name'] = i
#programme_df= programme_df[~programme_df['Course Name'].isin(['Statistical Mechanics','Thermodynamics'])]
condition = ((programme_df['Collection Name']=='Undergraduate (School of Physics and Astronomy) Level 9 and 10  courses')&
            (programme_df['Course Name']=='Thermal Physics'))
programme_df = programme_df[~condition]
#Informatics changes
a_inf, n_inf = filtering('Computer Science and Mathematics (BSc Hons)','School of Informatics',('MAT','Lev'))



for i in n_inf:
    for j in a_inf:
        if i in j:
            condition = ((courses_df['Module Name']==j))
            courses_df.loc[condition,'Module Name'] = i


name = 'Speech Processing, Speech Processing (Hons)'

condition = ((courses_df['Module Name']==name))
courses_df.loc[condition,'Module Name'] = 'Speech Processing (Hons)'

In [5]:
a_phi , n_phi = filtering('Philosophy and Mathematics (MA Hons)','School of Philosophy, Psychology and Language Sciences',('MAT'))

for i in n_phi:
    for j in a_phi:
        if i in j:
            condition = ((courses_df['Module Name']==j))
            courses_df.loc[condition,'Module Name'] = i
#Changing name of degree programmes
for degree in programmes:
    for year in years:
        condition = ((programme_df['Programme Name']==degree) &
                    (programme_df['Programme Year'] == year))
        programme_df.loc[condition, 'Programme Name'] = degree + ': Year ' + str(year)

programmes = programme_df['Programme Name'].unique()
#Filtering unnecessary courses
n_phy_3 = filtering('Mathematics and Physics (BSc Hons): Year 3','School of Physics and Astronomy',('Lev','MAT'))[1]
n_phy_4 = filtering('Mathematics and Physics (BSc Hons): Year 4','School of Physics and Astronomy',('Lev','MAT'))[1]
n_bus_3 = filtering('Mathematics and Business BSc (Hons): Year 3','Business School',('Lev','MAT'))[1]
n_bus_4 = filtering('Mathematics and Business BSc (Hons): Year 4','Business School',('Lev','MAT'))[1]
n_phi_3 = filtering('Philosophy and Mathematics (MA Hons): Year 3','School of Philosophy, Psychology and Language Sciences',('Lev','MAT'))[1]
n_phi_4 = filtering('Philosophy and Mathematics (MA Hons): Year 4','School of Philosophy, Psychology and Language Sciences',('Lev','MAT'))[1]
not_required_courses = n_phy_3+n_phy_4+n_bus_3+n_bus_4+n_phi_3+n_phi_4

programme_df = programme_df[
    ~programme_df['Course Name'].isin(not_required_courses)
]

In [6]:
#Fixing Reg Group Names
for degree in programmes:
    new = programme_df[programme_df['Programme Name']==degree]
    groups = new['Collection Reg Group'].dropna().unique()
    for collection in groups:
        condition = (
            (programme_df['Programme Name']==degree)&
            (programme_df['Collection Reg Group']==collection)
        )
        programme_df.loc[condition, 'Collection Reg Group'] = degree + ', Group ' + str(collection) 

condition =  (
    (programme_df['Collection Reg Group'].isna())&
    (programme_df['Programme Year'].isin([3,4]))
)
programme_df.loc[condition, [
    'Collection Reg Group',
    'Collection Group Max',
    'Collection Group Min'
]] = programme_df.loc[condition, [
    'Collection Name',
    'Collection Max Value',
    'Collection Min Value'
]].values

condition_1 = ((programme_df['Collection Reg Group']== 'Mathematics and Physics (BSc Hons): Year 3, Group A'))
programme_df.loc[condition_1,['Collection Group Min','Collection Group Max']]= [0.0,40.0]
condition_2 = ((programme_df['Collection Reg Group']== 'Mathematics and Business BSc (Hons): Year 3, Group B'))
programme_df.loc[condition_2,['Collection Group Min','Collection Group Max']]= [0,60.0]
condition_3 = ((programme_df['Collection Reg Group']== 'Philosophy and Mathematics (MA Hons): Year 3, Group B'))
programme_df.loc[condition_3,['Collection Group Min','Collection Group Max']]= [40.0,80.0]
condition_4 = ((programme_df['Collection Reg Group']== 'Economics and Mathematics (MA Hons): Year 3, Group A'))
programme_df.loc[condition_4,['Collection Group Min','Collection Group Max']]= [0.0,20.0]
condition_5 = ((programme_df['Collection Reg Group']== 'Mathematics and Physics (BSc Hons): Year 4, Group A'))
programme_df.loc[condition_5,['Collection Group Min','Collection Group Max']]= [0.0,20.0]
condition_6 = ((programme_df['Collection Reg Group']== 'Mathematics and Physics (BSc Hons): Year 4, Group B'))
programme_df.loc[condition_6,['Collection Group Min','Collection Group Max']]= [0.0,40.0]
condition_7 = ((programme_df['Collection Reg Group']== 'Mathematics and Business BSc (Hons): Year 4, Group A'))
programme_df.loc[condition_7,['Collection Group Min','Collection Group Max']]= [40.0,60.0]
condition_8 = ((programme_df['Collection Reg Group']== 'Computer Science and Mathematics (BSc Hons): Year 4, Group A'))
programme_df.loc[condition_8,['Collection Group Min','Collection Group Max']]= [0.0,40.0]
condition_9 = ((programme_df['Collection Reg Group']== 'Computer Science and Mathematics (BSc Hons): Year 4, Group B'))
programme_df.loc[condition_9,['Collection Group Min','Collection Group Max']]= [0.0,60.0]

In [7]:
courses_df = courses_df[courses_df['Event Type'].isin(['Lecture', 'Laboratory','Seminar'])]

### Extract timetable structure and information

In [8]:
def extract_programme_structure(programme_df):
    """
    Extract programme-specific information including collections and regression groups
    Returns a dictionary structure with collections and regression groups as dictionaries
    """
    # Get unique programmes
    programmes = programme_df['Programme Name'].unique().tolist()
    
    programme_data = {}
    
    for programme in programmes:
        prog_data = programme_df[programme_df['Programme Name'] == programme]
        
        # Get compulsory and optional courses for this programme
        compulsory_courses = prog_data[
            prog_data['Compulsory/Optional'] == 'Compulsory'
        ]['Course Name'].unique().tolist()
        
        optional_courses = prog_data[
            prog_data['Compulsory/Optional'] == 'Optional'
        ]['Course Name'].unique().tolist()
        
        # Get all courses for this programme
        all_courses = prog_data['Course Name'].unique().tolist()
        
        # Get collection information - store as dictionary
        collections = {}
        for _, row in prog_data.iterrows():
            collection_code = row.get('Collection Code')
            if pd.notna(collection_code) and collection_code != '':
                if collection_code not in collections:
                    collections[collection_code] = {
                        'name': row.get('Collection Name', ''),
                        'min_value': row.get('Collection Min Value', 0),
                        'max_value': row.get('Collection Max Value', 999),
                        'reg_group': row.get('Collection Reg Group', ''),
                        'courses': []
                    }
                # Add course to collection if not already there
                if row['Course Name'] not in collections[collection_code]['courses']:
                    collections[collection_code]['courses'].append(row['Course Name'])
        
        # Get regression group information - store as dictionary
        regression_groups = {}
        for _, row in prog_data.iterrows():
            reg_group = row.get('Collection Reg Group')
            if pd.notna(reg_group) and reg_group != '':
                if reg_group not in regression_groups:
                    regression_groups[reg_group] = {
                        'min_value': row.get('Collection Group Min', 0),
                        'max_value': row.get('Collection Group Max', 999),
                        'collections': []
                    }
                # Add collection to regression group if not already there
                collection_code = row.get('Collection Code')
                if pd.notna(collection_code) and collection_code != '':
                    if collection_code not in regression_groups[reg_group]['collections']:
                        regression_groups[reg_group]['collections'].append(collection_code)
        
        programme_data[programme] = {
            'compulsory_courses': compulsory_courses,
            'optional_courses': optional_courses,
            'all_courses': all_courses,
            'collections': collections,
            'regression_groups': regression_groups
        }
    
    return programme_data
programme_data = extract_programme_structure(programme_df)
#programme_data

In [9]:


def extract_timetable_info(courses_df, programmes_data):
    """
    Extract v parameter with dynamic duration per course
    """
    # Filter for all relevant courses
    all_relevant_courses = {"G1","G2","G3","G4",'G5','G6','G7','G8','O1','O2','O3','O4'}
    
    for programme in programmes_data:
        all_relevant_courses.update(programmes_data[programme]['compulsory_courses'])
        all_relevant_courses.update(programmes_data[programme]['optional_courses'])
    
    # Filter timetable data for relevant courses
    relevant_schedule = courses_df[courses_df['Module Name'].isin(all_relevant_courses)].copy()
    
    # Parse timeslot (format: "Monday 10:00")
    def parse_timeslot(timeslot):
        if pd.isna(timeslot):
            return None, None
        parts = str(timeslot).split()
        if len(parts) >= 2:
            day = parts[0]
            time_str = parts[1]
            hour = int(time_str.split(':')[0])
            return day, hour
        return None, None
    
    # Parse weeks (format like "1-11,13-24")
    def parse_weeks(weeks_str):
        if pd.isna(weeks_str):
            return []
        weeks = []
        for part in str(weeks_str).split(','):
            if '-' in part:
                start, end = map(int, part.split('-'))
                weeks.extend(range(start, end + 1))
            else:
                weeks.append(int(part))
        return weeks
    
    # Map days to integers
    day_map = {
        'Monday': 1, 'Tuesday': 2, 'Wednesday': 3, 'Thursday': 4, 
        'Friday': 5, 'Saturday': 6, 'Sunday': 7,
        'Mon': 1, 'Tue': 2, 'Wed': 3, 'Thu': 4, 'Fri': 5, 'Sat': 6, 'Sun': 7
    }
    
    # Create v parameter dictionary
    v = {}
    
    # For each course, build a dictionary of its scheduled events with durations
    for course in relevant_schedule['Module Name'].unique():
        course_schedule = relevant_schedule[relevant_schedule['Module Name'] == course]
        
        for _, row in course_schedule.iterrows():
            # Get timeslot
            day, start_hour = parse_timeslot(row['Timeslot'])
            if day is None:
                continue
                
            day_num = day_map.get(day, 0)
            if day_num == 0:
                print(f"Warning: Unknown day format: {day}")
                continue
            
            # Get duration from the Duration column
            duration = row['Duration (minutes)']
            if pd.isna(duration):
                # Fallback: use default based on event type
                event_type = row['Event Type']
                default_durations = {
                    'Lecture': 60,
                    'Workshop': 60,
                    'Tutorial': 60,
                    'Lab': 120,
                    'Seminar': 60
                }
                duration = default_durations.get(event_type, 60)
            
            # Calculate end hour (assuming duration is in minutes)
            # If duration is 60, end_hour = start_hour + 1
            # If duration is 90, end_hour = start_hour + 1.5 (we'll handle half-hours)
            # For simplicity, we'll treat hour slots as discrete and mark all hours covered
            duration_hours = duration / 60
            end_hour_float = start_hour + duration_hours
            
            # Determine which hour slots this event occupies
            # For integer hours, we can use integer hours
            occupied_hours = []
            current_hour = start_hour
            remaining_duration = duration
            
            while remaining_duration > 0:
                occupied_hours.append(current_hour)
                remaining_duration -= 60
                current_hour += 1
            
            # Alternative: if you want to handle half-hour slots more precisely
            # occupied_hours = [h for h in range(start_hour, start_hour + int(np.ceil(duration_hours)))]
            
            # Parse weeks
            weeks = parse_weeks(row['Weeks'])
            
            # Determine semester
            semester = row['Semester']
            if semester not in [1, 2]:
                # If semester is not 1 or 2, try to infer from weeks
                if weeks and max(weeks) <= 26:
                    semester = 1
                elif weeks and min(weeks) >= 27:
                    semester = 2
                else:
                    semester = 1
            
            campus = row['Campus']
            
            # For each week the course runs
            for week in weeks:
                # For each hour slot the course occupies
                for hour in occupied_hours:
                    # For each programme that includes this course
                    for programme in programmes_data:
                        if course in programmes_data[programme]['all_courses']:
                            v[(course, day_num, hour, week, programme, campus)] = 1
                        else:
                            v[(course, day_num, hour, week, programme, campus)] = 0
    
    return v
programmes_data = extract_programme_structure(programme_df)
#extract_timetable_info(courses_df, programmes_data)

In [10]:
def extract_days_hours_from_v(v):
    """
    Extract unique days and hours from v dictionary
    """
    if not v:
        # If v is empty, use default days and hours
        return [1, 2, 3, 4, 5], list(range(9, 17))  # Monday-Friday, 9am-5pm
    
    days = sorted(list(set([key[1] for key in v.keys()])))
    hours = sorted(list(set([key[2] for key in v.keys()])))
    
    # If no days/hours found, use defaults
    if not days:
        days = [1, 2, 3, 4, 5]
    if not hours:
        hours = list(range(9, 17))
    
    return days, hours

# Add this to your extract_all_parameters function
def extract_all_parameters(courses_df, programme_df):
    """
    Complete extraction including days and hours
    """
    
    # Step 1: Define Maths courses
    gateway_courses = ["G1","G2","G3","G4",'G5','G6','G7','G8']
    optional_courses = ['O1','O2','O3','O4']
    
    # Step 2: Extract programme structure
    programme_data = extract_programme_structure(programme_df)
    programmes = list(programme_data.keys())
    
    # Step 3: Extract timetable info
    v = extract_timetable_info(courses_df, programme_data)
    
    # Step 4: Extract days and hours from v
    #days, hours = extract_days_hours_from_v(v)
    days = [1,2,3,4,5]
    hours = [9,10,11,12,13,14,15,16,17]

    weeks = list(range(1, 53))  # E = {1, ..., 52}
    weeks_sem1 = list(range(9, 19))  # E_1 = {9, ..., 19}
    weeks_sem2 = list(range(26, 37))  # E_2 = {26, ..., 37}

    campuses = courses_df['Campus'].dropna().unique().tolist()
    
    # Build the complete parameter set
    parameters = {
        # Sets
        'G': gateway_courses,
        'O': optional_courses,
        'S': [1, 2],  # Semesters
        'D': days,    # Days extracted from timetable
        'H': hours,   # Hours extracted from timetable
        'W': [1, 2],  # Week parity
        'Q': programmes,
        'E': weeks,           # All weeks
        'E1': weeks_sem1,     # Semester 1 weeks
        'E2': weeks_sem2,     # Semester 2 weeks
        'K': campuses,        # Campuses
        # Parameters
        'R_g_L': 3,  # Fixed for gateway
        'R_g_W': 1,  # Fixed for gateway
        'R_g_F': 1,  # Fixed for gateway
        'R_o_L': 3,  # Fixed for optional
        'R_o_W': 1,  # Fixed for optional
        'C_g': 4,    # 4 gateway courses per semester
        'C_o': 2,    # 2 optional courses per semester
        'n_q': {programme: 40 for programme in programmes},  # 40 credits outside Maths
        'n_co': {},  # Credits for each course
        'v': v,
        
        # Programme-specific data
        'programme_data': programme_data,
        
        # Collection and regression groups
        'min_CL': {},
        'max_CL': {},
        'min_CR': {},
        'max_CR': {},
        'SC_cl': {},
        'SCL_cr': {}
    }
    
    # Extract credits (same as before)
    all_courses = set(gateway_courses + optional_courses)
    for programme in programmes:
        all_courses.update(programme_data[programme]['all_courses'])
    
    for course in all_courses:
        credit_info = programme_df[programme_df['Course Name'] == course]['SCQF Credits']
        if len(credit_info) > 0:
            credit_value = credit_info.iloc[0]
            if isinstance(credit_value, str):
                import re
                numbers = re.findall(r'\d+\.?\d*', credit_value)
                if numbers:
                    parameters['n_co'][course] = float(numbers[0])
                else:
                    parameters['n_co'][course] = 20.0
            else:
                parameters['n_co'][course] = float(credit_value)
        else:
            parameters['n_co'][course] = 20.0
    
    # Extract collection information
    for programme in programmes:
        prog_collections = programme_data[programme]['collections']
        for cl_id, cl_info in prog_collections.items():
            parameters['min_CL'][cl_id] = cl_info['min_value']
            parameters['max_CL'][cl_id] = cl_info['max_value']
            
            if cl_id not in parameters['SC_cl']:
                parameters['SC_cl'][cl_id] = []
            for course in cl_info['courses']:
                if course not in parameters['SC_cl'][cl_id]:
                    parameters['SC_cl'][cl_id].append(course)
    
    # Extract regression group information
    for programme in programmes:
        prog_reg_groups = programme_data[programme]['regression_groups']
        for rg_id, rg_info in prog_reg_groups.items():
            parameters['min_CR'][rg_id] = rg_info['min_value']
            parameters['max_CR'][rg_id] = rg_info['max_value']
            
            if rg_id not in parameters['SCL_cr']:
                parameters['SCL_cr'][rg_id] = []
            for cl in rg_info['collections']:
                if cl not in parameters['SCL_cr'][rg_id]:
                    parameters['SCL_cr'][rg_id].append(cl)
    
    # Print summary
    print(f"\nExtraction Summary:")
    print(f"  - Gateway courses: {len(parameters['G'])}")
    print(f"  - Optional courses: {len(parameters['O'])}")
    print(f"  - Programmes: {len(parameters['Q'])}")
    print(f"  - Days: {parameters['D']}")
    print(f"  - Hours: {parameters['H']}")
    print(f"  - Weeks: {len(parameters['E'])} (Sem1: {len(parameters['E1'])}, Sem2: {len(parameters['E2'])})")
    print(f"  - Campuses: {parameters['K']}")
    print(f"  - Collections: {len(parameters['SC_cl'])}")
    print(f"  - Regression groups: {len(parameters['SCL_cr'])}")
    print(f"  - Timetable entries: {len(parameters['v'])}")
    print(f"  - Courses with credits: {len(parameters['n_co'])}")
    
    return parameters
parameters = extract_all_parameters(courses_df,programme_df)


Extraction Summary:
  - Gateway courses: 8
  - Optional courses: 4
  - Programmes: 10
  - Days: [1, 2, 3, 4, 5]
  - Hours: [9, 10, 11, 12, 13, 14, 15, 16, 17]
  - Weeks: 52 (Sem1: 10, Sem2: 11)
  - Campuses: ['Central', 'Holyrood', 'New College', 'Kings Buildings', 'Lauriston']
  - Collections: 22
  - Regression groups: 17
  - Timetable entries: 57810
  - Courses with credits: 253


### Generate the data for the model

In [11]:
def create_model_data(parameters):
    """
    Convert extracted parameters into model-ready format
    """
    
    # Create mappings for indices
    course_to_idx = {course: idx for idx, course in enumerate(parameters['G'] + parameters['O'])}
    programme_to_idx = {prog: idx for idx, prog in enumerate(parameters['Q'])}
    
    # Create day and hour mappings
    # You'll need to extract actual days and hours from your timetable data
    days = sorted(list(set([key[1] for key in parameters['v'].keys() if len(key) > 1])))
    hours = sorted(list(set([key[2] for key in parameters['v'].keys() if len(key) > 2])))
    
    model_ready = {
        'G': parameters['G'],
        'O': parameters['O'],
        'S': parameters['S'],
        'D': days,
        'H': hours,
        'W': [1, 2],
        'Q': parameters['Q'],
        'E': parameters['E'],      # All weeks
        'E1': parameters['E1'],    # Semester 1 weeks
        'E2': parameters['E2'],    # Semester 2 weeks
        'K': parameters['K'],      # Campuses
        
        'R_g_L': parameters['R_g_L'],
        'R_g_W': parameters['R_g_W'],
        'R_g_F': parameters['R_g_F'],
        'R_o_L': parameters['R_o_L'],
        'R_o_W': parameters['R_o_W'],
        'C_g': parameters['C_g'],
        'C_o': parameters['C_o'],
        
        'min_CL': parameters['min_CL'],
        'max_CL': parameters['max_CL'],
        'min_CR': parameters['min_CR'],
        'max_CR': parameters['max_CR'],
        
        'n_co': parameters['n_co'],
        'n_q': parameters['n_q'],
        
        'CO_CO_q': {prog: parameters['programme_data'][prog]['compulsory_courses'] 
                   for prog in parameters['Q']},
        'CO_OP_q': {prog: parameters['programme_data'][prog]['optional_courses'] 
                   for prog in parameters['Q']},
        'CO_q': {prog: parameters['programme_data'][prog]['all_courses'] 
                for prog in parameters['Q']},
        
        'SC_cl': parameters['SC_cl'],
        'SCL_cr': parameters['SCL_cr'],
        
        'v': parameters['v']
    }
    
    return model_ready

In [12]:
# Extract all parameters
parameters = extract_all_parameters(courses_df, programme_df)

# Convert to model-ready format
model_data = create_model_data(parameters)

# Now you can use model_data to build your Xpress model
print(f"Number of gateway courses: {len(model_data['G'])}")
print(f"Number of optional courses: {len(model_data['O'])}")
print(f"Number of programmes: {len(model_data['Q'])}")
print(f"Collection groups: {len(model_data['SC_cl'])}")
print(f"Regression groups: {len(model_data['SCL_cr'])}")


Extraction Summary:
  - Gateway courses: 8
  - Optional courses: 4
  - Programmes: 10
  - Days: [1, 2, 3, 4, 5]
  - Hours: [9, 10, 11, 12, 13, 14, 15, 16, 17]
  - Weeks: 52 (Sem1: 10, Sem2: 11)
  - Campuses: ['Central', 'Holyrood', 'New College', 'Kings Buildings', 'Lauriston']
  - Collections: 22
  - Regression groups: 17
  - Timetable entries: 57810
  - Courses with credits: 253
Number of gateway courses: 8
Number of optional courses: 4
Number of programmes: 10
Collection groups: 22
Regression groups: 17


In [13]:

travel_data = [
    ("Bioquarter", "Bioquarter", 0),
    ("Central", "Bioquarter", 60),
    ("Easter Bush", "Bioquarter", 60),
    ("Holyrood", "Bioquarter", 60),
    ("King's Buildings", "Bioquarter", 60),
    ("Lauriston", "Bioquarter", 60),
    ("New College", "Bioquarter", 60),
    ("Western General", "Bioquarter", 60),
    ("Bioquarter", "Central", 60),
    ("Central", "Central", 0),
    ("Easter Bush", "Central", 60),
    ("Holyrood", "Central", 10),
    ("King's Buildings", "Central", 30),
    ("Lauriston", "Central", 10),
    ("New College", "Central", 10),
    ("Western General", "Central", 60),
    ("Bioquarter", "Easter Bush", 60),
    ("Central", "Easter Bush", 60),
    ("Easter Bush", "Easter Bush", 0),
    ("Holyrood", "Easter Bush", 60),
    ("King's Buildings", "Easter Bush", 60),
    ("Lauriston", "Easter Bush", 60),
    ("New College", "Easter Bush", 60),
    ("Western General", "Easter Bush", 60),
    ("Bioquarter", "Holyrood", 60),
    ("Central", "Holyrood", 10),
    ("Easter Bush", "Holyrood", 60),
    ("Holyrood", "Holyrood", 0),
    ("King's Buildings", "Holyrood", 30),
    ("Lauriston", "Holyrood", 10),
    ("New College", "Holyrood", 10),
    ("Western General", "Holyrood", 60),
    ("Bioquarter", "King's Buildings", 60),
    ("Central", "King's Buildings", 30),
    ("Easter Bush", "King's Buildings", 60),
    ("Holyrood", "King's Buildings", 30),
    ("King's Buildings", "King's Buildings", 0),
    ("Lauriston", "King's Buildings", 30),
    ("New College", "King's Buildings", 30),
    ("Western General", "King's Buildings", 60),
    ("Bioquarter", "Lauriston", 60),
    ("Central", "Lauriston", 10),
    ("Easter Bush", "Lauriston", 60),
    ("Holyrood", "Lauriston", 10),
    ("King's Buildings", "Lauriston", 30),
    ("Lauriston", "Lauriston", 0),
    ("New College", "Lauriston", 10),
    ("Western General", "Lauriston", 60),
    ("Bioquarter", "New College", 60),
    ("Central", "New College", 10),
    ("Easter Bush", "New College", 60),
    ("Holyrood", "New College", 10),
    ("King's Buildings", "New College", 30),
    ("Lauriston", "New College", 10),
    ("New College", "New College", 0),
    ("Western General", "New College", 60),
    ("Bioquarter", "Western General", 60),
    ("Central", "Western General", 60),
    ("Easter Bush", "Western General", 60),
    ("Holyrood", "Western General", 60),
    ("King's Buildings", "Western General", 60),
    ("Lauriston", "Western General", 60),
    ("New College", "Western General", 60),
    ("Western General", "Western General", 0)
]

# Create a dictionary for fast lookup
travel_time_dict = {}
for from_camp, to_camp, tt in travel_data:
    travel_time_dict[(from_camp, to_camp)] = tt


## Build model from parameters

In [14]:
def build_model_from_parameters(parameters, soft_upper_bounds):
    """
    Build the full Xpress model for joint timetabling with soft constraints,
    then add per‑curriculum upper bounds on each soft objective,
    and set the objective to maximise the minimum number of gateways taken by any curriculum.

    Parameters:
    - parameters: the full parameters dict from extract_all_parameters
    - soft_upper_bounds: dict with keys: 'late', 'lunch', 'isolated', 'days', 'wednesday', 'clash',
                         each mapping to a dict {q: value} giving the maximum allowed raw count.

    Returns:
    - p: Xpress problem
    - constraint_list: list of (row_index, name)
    - var_dicts: dictionary of variable dictionaries
    """
    # ==================== UNPACK PARAMETERS ====================
    G = parameters['G']
    O = parameters['O']
    S = parameters['S']
    D = parameters['D']
    H = parameters['H']
    W = parameters['W']
    Q = parameters['Q']
    E = parameters['E']
    E1 = parameters['E1']
    E2 = parameters['E2']
    K = parameters['K']
    C_g = parameters['C_g']
    C_o = parameters['C_o']
    R_g_L = parameters['R_g_L']
    R_g_W = parameters['R_g_W']
    R_g_F = parameters['R_g_F']
    R_o_L = parameters['R_o_L']
    R_o_W = parameters['R_o_W']
    v = parameters['v']
    programme_data = parameters['programme_data']
    CO_CO_q = {prog: programme_data[prog]['compulsory_courses'] for prog in Q}
    CO_OP_q = {prog: programme_data[prog]['optional_courses'] for prog in Q}
    CO_q = {prog: programme_data[prog]['all_courses'] for prog in Q}
    SC_cl = parameters['SC_cl']
    SCL_cr = parameters['SCL_cr']
    min_CL = parameters['min_CL']
    max_CL = parameters['max_CL']
    min_CR = parameters['min_CR']
    max_CR = parameters['max_CR']
    n_co = parameters['n_co']
    n_q = parameters['n_q']
    QQ=Q+['math']
    # Soft constraint weights (not used in objective, but kept for completeness)
    lambda_travel = 0.01
    lambda_clash = 0.1
    lambda_late     = 0.344249   # after 5pm
    lambda_lunch    = 0.112523   # lunch time
    lambda_isolated = 0.057430   # isolated class
    lambda_days     = 0.133207   # number of days
    lambda_wed      = 0.028978 
    # Travel time dictionary
    global travel_time_dict
    all_top_courses = [
        "MATH10065", "MATH08064", "MATH08063", "MATH08051",
        "MATH08066", "MATH08059", "MATH08058", "MATH10067"
    ]
    course_to_g = {course: f"G{i+1}" for i, course in enumerate(sorted(all_top_courses))}

    # Top courses per (programme, year) – as extracted from data
    top_courses_data = {
        ("UTBSCMATBU1F", 3): ["MATH10065", "MATH08064", "MATH08063", "MATH08051"],
        ("UTBSCMATBU1F", 4): ["MATH10065", "MATH08064", "MATH08063", "MATH08051"],
        ("UTCMPMA", 3): ["MATH08064", "MATH08063", "MATH08066", "MATH08059"],
        ("UTCMPMA", 4): ["MATH08064", "MATH08063", "MATH08066", "MATH08059"],
        ("UTECNMA", 3): ["MATH08051", "MATH08064", "MATH08063", "MATH08066"],
        ("UTECNMA", 4): ["MATH08051", "MATH08064", "MATH08063", "MATH08066"],
        ("UTMATPH", 3): ["MATH08066", "MATH08064", "MATH08058", "MATH08059"],
        ("UTMATPH", 4): ["MATH08066", "MATH08064", "MATH08058", "MATH08059"],
        ("UTPHIMA", 3): ["MATH08063", "MATH08064", "MATH08059", "MATH10067"],
        ("UTPHIMA", 4): ["MATH08063", "MATH08064", "MATH08059", "MATH10067"],
    }

    # Map programme base code to full curriculum name (as appears in Q)
    prog_name_map = {
        "UTECNMA": "Economics and Mathematics (MA Hons)",
        "UTBSCMATBU1F": "Mathematics and Business BSc (Hons)",
        "UTCMPMA": "Computer Science and Mathematics (BSc Hons)",
        "UTMATPH": "Mathematics and Physics (BSc Hons)",
        "UTPHIMA": "Philosophy and Mathematics (MA Hons)"
    }

    pathways = {}
    for (prog_base, year), courses in top_courses_data.items():
        full_name = f"{prog_name_map[prog_base]}: Year {int(year)}"
        pathways[full_name] = [course_to_g[c] for c in courses]   
    # Constants for soft constraints based on actual hours
    late_hour = 17
    lunch_hours = [12, 13]
    isolated_hours = list(range(10, 17))
    wednesday_hours = list(range(13, 18))
    wed_idx = 3

    # Create problem
    p = xp.problem()

    # ==================== DECISION VARIABLES ====================
    print("Creating decision variables...")

    # Gateway variables
    x_L = {}
    x_W = {}
    x_F = {}
    for g in G:
        for d in D:
            for h in H:
                for s in S:
                    x_L[(g, d, h, s)] = xp.var(vartype=xp.binary, name=f"xL_{g}_{d}_{h}_{s}")
                    x_W[(g, d, h, s)] = xp.var(vartype=xp.binary, name=f"xW_{g}_{d}_{h}_{s}")
                    for w in W:
                        x_F[(g, d, h, s, w)] = xp.var(vartype=xp.binary, name=f"xF_{g}_{d}_{h}_{s}_{w}")

    # Optional course variables
    y_L = {}
    y_W = {}
    for o in O:
        for d in D:
            for h in H:
                for s in S:
                    y_L[(o, d, h, s)] = xp.var(vartype=xp.binary, name=f"yL_{o}_{d}_{h}_{s}")
                    y_W[(o, d, h, s)] = xp.var(vartype=xp.binary, name=f"yW_{o}_{d}_{h}_{s}")

    # Semester assignment
    z = {(g, s): xp.var(vartype=xp.binary, name=f"z_{g}_{s}") for g in G for s in S}
    w_var = {(o, s): xp.var(vartype=xp.binary, name=f"w_{o}_{s}") for o in O for s in S}

    # Student choice for non‑maths courses
    a = {}
    for q in Q:
        for co in CO_q[q]:
            a[(q, co)] = xp.var(vartype=xp.binary, name=f"a_{q}_{co}")

    # Gateway pathway choice for each curriculum
    ta = {}
    for g in G:
        for s in S:
            for q in Q:
                ta[(g, s, q)] = xp.var(vartype=xp.binary, name=f"ta_{g}_{s}_{q}")

    # Linearisation of ta * x_L
    lin = {}
    for g in G:
        for d in D:
            for h in H:
                for s in S:
                    for q in Q:
                        lin[(g, d, h, s, q)] = xp.var(vartype=xp.binary, name=f"lin_{g}_{d}_{h}_{s}_{q}")

    # Auxiliary for travel time (triple product a * ta * x_L)
    ind = {}
    for q in Q:
        for co in CO_q[q]:
            for g in G:
                for d in D:
                    for h in H:
                        for s in S:
                            ind[(q, co, g, d, h, s)] = xp.var(vartype=xp.binary, name=f"ind_{q}_{co}_{g}_{d}_{h}_{s}")

    # Auxiliary for product of two a's (for travel between two non‑maths courses)
    ch = {}
    for q in Q:
        for co in CO_q[q]:
            for co2 in CO_q[q]:
                if co < co2:
                    ch[(q, co, co2)] = xp.var(vartype=xp.binary, name=f"ch_{q}_{co}_{co2}")

    # Isolated class indicators
    is_isolated = {}
    for q in QQ:
        for d in D:
            for h in isolated_hours:
                for s in S:
                    weeks = E1 if s == 1 else E2
                    for e in weeks:
                        is_isolated[(q, d, h, s, e)] = xp.var(vartype=xp.binary, name=f"is_isolated_{q}_{d}_{h}_{s}_{e}")

    # Day‑used indicators
    b = {}
    for q in QQ:
        for d in D:
            for s in S:
                weeks = E1 if s == 1 else E2
                for e in weeks:
                    b[(q, d, s, e)] = xp.var(vartype=xp.binary, name=f"b_{q}_{d}_{s}_{e}")

    # Variable for the minimum number of gateways taken by any curriculum
    gwn = xp.var(vartype=xp.integer, name="gwn")


    # Add all variables
    all_vars = (list(x_L.values()) + list(x_W.values()) + list(x_F.values()) +
                list(y_L.values()) + list(y_W.values()) +
                list(z.values()) + list(w_var.values()) +
                list(a.values()) + list(ta.values()) + list(lin.values()) +
                list(ind.values()) + list(ch.values()) +
                list(is_isolated.values()) + list(b.values()) + [gwn])
    p.addVariable(all_vars)

    # ==================== HARD CONSTRAINTS ====================
    print("Adding hard constraints...")
    constraint_count = 0
    constraint_list = []

    def add_constraint(expr, name):
        nonlocal constraint_count
        row = p.addConstraint(expr)
        constraint_list.append((row, name))
        constraint_count += 1

    def flatten_weeks(week_list):
        if not isinstance(week_list, (list, tuple)):
            return [week_list]
        flat = []
        for item in week_list:
            if isinstance(item, (list, tuple)):
                flat.extend(flatten_weeks(item))
            else:
                flat.append(item)
        return flat

    E_flat = flatten_weeks(E)
    E1_flat = flatten_weeks(E1)
    E2_flat = flatten_weeks(E2)
    p.addConstraint(gwn == 8)

    # 1. Fortnightly workshops per gateway
    for g in G:
        add_constraint(xp.Sum(x_F[(g, d, h, s, w)] for d in D for h in H for s in S for w in W) == R_g_F,
                       f"Fortnightly_{g}")

    # 2. No multiple events per course in same slot
    for g in G:
        for d in D:
            for h in H:
                for s in S:
                    add_constraint(x_L[(g, d, h, s)] + x_W[(g, d, h, s)] +
                                   xp.Sum(x_F[(g, d, h, s, w)] for w in W) <= 1,
                                   f"CourseNoClash_G_{g}_{d}_{h}_{s}")
    for o in O:
        for d in D:
            for h in H:
                for s in S:
                    add_constraint(y_L[(o, d, h, s)] + y_W[(o, d, h, s)] <= 1,
                                   f"CourseNoClash_O_{o}_{d}_{h}_{s}")

    # 3. Global no‑clash per week (Maths only)
    for d in D:
        for h in H:
            for s in S:
                for w in W:
                    add_constraint(
                        xp.Sum(x_L[(g, d, h, s)]for g in G) +
                        xp.Sum(y_L[(o, d, h, s)]  for o in O)  <= 1,
                        f"GlobalNoClash_{d}_{h}_{s}_{w}")

    # 4. Events limited to assigned semester
    for g in G:
        for d in D:
            for h in H:
                for s in S:
                    add_constraint(x_L[(g, d, h, s)] <= z[(g, s)],
                                   f"SemLimit_xL_{g}_{d}_{h}_{s}")
                    add_constraint(x_W[(g, d, h, s)] <= z[(g, s)],
                                   f"SemLimit_xW_{g}_{d}_{h}_{s}")
                    for w in W:
                        add_constraint(x_F[(g, d, h, s, w)] <= z[(g, s)],
                                       f"SemLimit_xF_{g}_{d}_{h}_{s}_{w}")
    for o in O:
        for d in D:
            for h in H:
                for s in S:
                    add_constraint(y_L[(o, d, h, s)] <= w_var[(o, s)],
                                   f"SemLimit_yL_{o}_{d}_{h}_{s}")
                    add_constraint(y_W[(o, d, h, s)] <= w_var[(o, s)],
                                   f"SemLimit_yW_{o}_{d}_{h}_{s}")

    # 5. Gateway course semester assignment
    for g in G:
        add_constraint(xp.Sum(z[(g, s)] for s in S) == 1, f"GateSemAssign_{g}")

    # 6. Optional course semester assignment
    for o in O:
        add_constraint(xp.Sum(w_var[(o, s)] for s in S) == 1, f"OptSemAssign_{o}")

    # 7. Exactly required number of courses per semester
    for s in S:
        add_constraint(xp.Sum(z[(g, s)] for g in G) == C_g, f"GateCount_{s}")
        add_constraint(xp.Sum(w_var[(o, s)] for o in O) == C_o, f"OptCount_{s}")

    # 8. Teaching event counts for gateway
    for g in G:
        for s in S:
            lect_count = R_g_L[g] if isinstance(R_g_L, dict) else R_g_L
            add_constraint(xp.Sum(x_L[(g, d, h, s)] for d in D for h in H) == lect_count * z[(g, s)],
                           f"GateLectures_{g}_{s}")
            workshop_count = R_g_W[g] if isinstance(R_g_W, dict) else R_g_W
            add_constraint(xp.Sum(x_W[(g, d, h, s)] for d in D for h in H) == workshop_count * z[(g, s)],
                           f"GateWorkshops_{g}_{s}")

    # 9. Teaching event counts for optional
    for o in O:
        for s in S:
            add_constraint(xp.Sum(y_L[(o, d, h, s)] for d in D for h in H) == R_o_L * w_var[(o, s)],
                           f"OptLectures_{o}_{s}")
            add_constraint(xp.Sum(y_W[(o, d, h, s)] for d in D for h in H) == R_o_W * w_var[(o, s)],
                           f"OptWorkshops_{o}_{s}")

    # 10. Collection credit requirements
    for cl_id in SC_cl:
        for q in Q:
            if cl_id in programme_data[q].get('collections', {}):
                collection_sum = xp.Sum(n_co[co] * a[(q, co)]
                                       for co in SC_cl[cl_id]
                                       if co in CO_q[q])
                add_constraint(collection_sum >= min_CL[cl_id],
                               f"CollectionMin_{cl_id}_{q}")
                add_constraint(collection_sum <= max_CL[cl_id],
                               f"CollectionMax_{cl_id}_{q}")

    # 11. Regression group credit requirements
    for rg_id in SCL_cr:
        for q in Q:
            if rg_id in programme_data[q].get('regression_groups', {}):
                reg_sum = xp.Sum(n_co[co] * a[(q, co)]
                                for cl_id in SCL_cr[rg_id]
                                for co in SC_cl[cl_id]
                                if co in CO_q[q])
                add_constraint(reg_sum >= min_CR[rg_id],
                               f"RegGroupMin_{rg_id}_{q}")
                add_constraint(reg_sum <= max_CR[rg_id],
                               f"RegGroupMax_{rg_id}_{q}")

    # 12. Compulsory courses must be taken
    for q in Q:
        for co in CO_CO_q[q]:
            add_constraint(a[(q, co)] == 1, f"Compulsory_{q}_{co}")

    # 13. Minimum credits outside Maths
    for q in Q:
        outside_credits = xp.Sum(n_co[co] * a[(q, co)]
                                for co in CO_q[q]
                                if co not in G and co not in O)
        add_constraint(outside_credits >= n_q[q], f"OutsideCredits_{q}")

    # 14. Linearisation for ta * xL (lin)
    for g in G:
        for d in D:
            for h in H:
                for s in S:
                    for q in Q:
                        add_constraint(lin[(g, d, h, s, q)] <= ta[(g, s, q)],
                                       f"Lin1_{g}_{d}_{h}_{s}_{q}")
                        add_constraint(lin[(g, d, h, s, q)] <= x_L[(g, d, h, s)],
                                       f"Lin2_{g}_{d}_{h}_{s}_{q}")
                        add_constraint(lin[(g, d, h, s, q)] >= ta[(g, s, q)] + x_L[(g, d, h, s)] - 1,
                                       f"Lin3_{g}_{d}_{h}_{s}_{q}")

    # 15. No clashes with compulsory courses per curriculum
    for q in Q:
        for d in D:
            for h in H:
                for s in S:
                    weeks = E1_flat if s == 1 else E2_flat if s == 2 else E_flat
                    for e in weeks:
                        if isinstance(e, (list, tuple)): continue
                        for k in K:
                            gateway_sum = xp.Sum(lin[(g, d, h, s, q)] for g in G)
                            compulsory_terms = []
                            for co in CO_CO_q[q]:
                                key = (co, d, h, e, q, k)
                                val = v.get(key, 0)
                                if val > 0:
                                    compulsory_terms.append(a[(q, co)] * val)
                            if compulsory_terms:
                                compulsory_sum = xp.Sum(compulsory_terms)
                                add_constraint(gateway_sum + compulsory_sum <= 1,
                                               f"ClashComp_{q}_{d}_{h}_{s}_{e}_{k}")

    # 16. No clashes with optional courses per curriculum
    for q in Q:
        for d in D:
            for h in H:
                for s in S:
                    weeks = E1_flat if s == 1 else E2_flat if s == 2 else E_flat
                    for e in weeks:
                        if isinstance(e, (list, tuple)): continue
                        for k in K:
                            gateway_sum = xp.Sum(lin[(g, d, h, s, q)] for g in G)
                            optional_terms = []
                            for co in CO_OP_q[q]:
                                key = (co, d, h, e, q, k)
                                val = v.get(key, 0)
                                if val > 0:
                                    optional_terms.append(a[(q, co)] * val)
                            if optional_terms:
                                optional_sum = xp.Sum(optional_terms)
                                add_constraint(gateway_sum + optional_sum <= 1,
                                               f"ClashOpt_{q}_{d}_{h}_{s}_{e}_{k}")
        # Pathway constraints
    for q in Q:
        for g in pathways[q]:
            p.addConstraint(
                xp.Sum(ta[g,s,q] for s in S) == 1)

    # 17. Minimum gateway courses per curriculum (at least 2 per semester) – we will replace with gwn constraint
    # Instead, we add constraints linking gwn to each curriculum's total gateways
    
    for q in Q:
        add_constraint(xp.Sum(ta[(g, s, q)] for g in G for s in S) >= gwn, f"GWN_{q}")
    
    #18 Exactly 4 gateway courses chosen across the year
    #for q in Q:
    #    add_constraint(xp.Sum(ta[(g, s, q)] for g in G for s in S ) == 4,
    #                       f"MinGateways_{q}")
    #     
    # 19 Run econ and business , and philosophy and computer science together, exclude them:
    # for g in G:
    #     for s in S:
    #         add_constraint(ta[(g,s, 'Economics and Mathematics (MA Hons): Year 3')] == ta[g,s, 'Mathematics and Business BSc (Hons): Year 3'], f"Business_and_Econ_{g}_{s}")
    # 
    #         add_constraint(ta[(g,s, 'Economics and Mathematics (MA Hons): Year 4')] == ta[g,s, 'Mathematics and Business BSc (Hons): Year 4'], f"Business_and_Econ_{g}_{s}")
    # 
    #         add_constraint(ta[(g,s, 'Philosophy and Mathematics (MA Hons): Year 3')] == ta[g,s, 'Computer Science and Mathematics (BSc Hons): Year 3'], f"Phil_and_CompSci_{g}_{s}")
    #         add_constraint(ta[(g,s, 'Philosophy and Mathematics (MA Hons): Year 4')] == ta[g,s, 'Computer Science and Mathematics (BSc Hons): Year 4'], f"Phil_and_CompSci_{g}_{s}")

    #20 Link ta to z: a curriculum can only take a gateway in a semester where the gateway runs
    for g in G:
        for s in S:
            for q in Q:
                add_constraint(ta[(g, s, q)] <= z[(g, s)], f"Ta_Z_{g}_{s}_{q}")
                

    # ==================== SOFT CONSTRAINT AUXILIARY ====================
    # Event presence P for all (q,d,h,s,e)
    # -------------------------------------------------------------------
# Event presence expressions
# -------------------------------------------------------------------

    q_math = 'math'

    print('QQ:',QQ)
    # P[(q,d,h,s,e)] will store the event-presence expression
    P = {}
    
    # --- Pure Mathematics curriculum ---
    for d in D:
        for h in H:
            for s in S:
                    weeks = E1 if s == 1 else E2
                    for e in weeks:

                        P[(q_math, d, h, s, e)] = (
                            xp.Sum(
                                x_L[(g, d, h, s)] 
                        
                                for g in G
                            ) +
                            xp.Sum(
                                y_L[(o, d, h, s)]                             for o in O
                            )
                        )
    
    # --- Joint curricula ---
    for q in QQ:
        if q == q_math:
            continue
        
        for d in D:
            for h in H:
                for s in S:
                    for e in E1 + E2:
                        P[(q, d, h, s, e)] = (
                            xp.Sum(lin[(g, d, h, s,q)] for g in G) +
                            xp.Sum(
                                v.get((co, d, h, e, q, k), 0) * a[(q, co)]
                                for co in CO_q[q]
                                for k in K
                            )
                        )
 
    # 18. Isolated classes
    for q in QQ:
        for d in D:
            for s in S:
                weeks = E1 if s == 1 else E2
                for e in weeks:
                    for h in isolated_hours:
                        p_curr = P[(q, d, h, s, e)]
                        p_next = P[(q, d, h+1, s, e)] if h+1 in H else 0
                        p_prev = P[(q, d, h-1, s, e)] if h-1 in H else 0
                        add_constraint(is_isolated[(q, d, h, s, e)] >= p_curr - p_next - p_prev,
                                       f"IsolatedGe_{q}_{d}_{h}_{s}_{e}")
                        add_constraint(is_isolated[(q, d, h, s, e)] <= p_curr,
                                       f"IsolatedLe_{q}_{d}_{h}_{s}_{e}")

    # 19. Days used
    for q in QQ:
        for d in D:
            for s in S:
                weeks = E1 if s == 1 else E2
                for e in weeks:
                    total_day = xp.Sum(P[(q, d, h, s, e)] for h in H)
                    add_constraint(b[(q, d, s, e)] <= total_day, f"DaysLe_{q}_{d}_{s}_{e}")
                    add_constraint(total_day <= len(H) * b[(q, d, s, e)], f"DaysGe_{q}_{d}_{s}_{e}")

    # 20. Linearisation for ind
    for q in Q:
        for co in CO_q[q]:
            for g in G:
                for d in D:
                    for h in H:
                        for s in S:
                            vv = ind[(q, co, g, d, h, s)]
                            add_constraint(vv <= a[(q, co)], f"Ind1_{q}_{co}_{g}_{d}_{h}_{s}")
                            add_constraint(vv <= ta[(g, s, q)], f"Ind2_{q}_{co}_{g}_{d}_{h}_{s}")
                            add_constraint(vv <= x_L[(g, d, h, s)], f"Ind3_{q}_{co}_{g}_{d}_{h}_{s}")
                            add_constraint(vv >= a[(q, co)] + ta[(g, s, q)] + x_L[(g, d, h, s)] - 2,
                                           f"Ind4_{q}_{co}_{g}_{d}_{h}_{s}")

    # 21. Linearisation for ch
    for q in Q:
        for co in CO_q[q]:
            for co2 in CO_q[q]:
                if co < co2:
                    vv = ch[(q, co, co2)]
                    add_constraint(vv <= a[(q, co)], f"Ch1_{q}_{co}_{co2}")
                    add_constraint(vv <= a[(q, co2)], f"Ch2_{q}_{co}_{co2}")
                    add_constraint(vv >= a[(q, co)] + a[(q, co2)] - 1, f"Ch3_{q}_{co}_{co2}")

    # ==================== PER‑CURRICULUM RAW SOFT EXPRESSIONS ====================
    # Compute raw expressions for late, lunch, wednesday, isolated, days, clash    # ==================== GLOBAL RAW SOFT EXPRESSIONS (SUM OVER ALL Q) ====================
    # We build one single expression per soft term that already includes all curricula
    # ==================== GLOBAL RAW SOFT EXPRESSIONS (SUM OVER ALL Q) ====================
    # 
    # late_expr = xp.Sum(P[(q, d, late_hour, s, e)] for q in QQ for s in S for e in (E1 if s==1 else E2) for d in D)
    # lunch_expr = xp.Sum(P[(q, d, h, s, e)] for q in QQ for s in S for e in (E1 if s==1 else E2) for d in D for h in lunch_hours)
    # wed_expr = xp.Sum(P[(q, d, h, s, e)] for q in QQ for s in S for e in (E1 if s==1 else E2) for d in D if d == wed_idx for h in wednesday_hours)
    # isolated_expr = xp.Sum(is_isolated[(q, d, h, s, e)] for q in QQ for s in S for e in (E1 if s==1 else E2) for d in D for h in isolated_hours)
    # days_expr = xp.Sum(b[(q, d, s, e)] for q in QQ for s in S for e in (E1 if s==1 else E2) for d in D)
    # 
    # # Clash expression (only joint curricula)
    # clash_expr = 0
    # for q in Q:   # Q does NOT include 'math'
    #     for s in S:
    #         weeks = E1 if s == 1 else E2
    #         for e in weeks:
    #             for d in D:
    #                 for h in H:
    #                     optional_maths = xp.Sum(y_L[(o, d, h, s)] for o in O)
    #                     # Pre‑compute constant sum over compulsory non‑maths courses
    #                     nonmaths_const = 0
    #                     for co in CO_CO_q[q]:
    #                         if co not in G and co not in O:
    #                             for k in K:
    #                                 nonmaths_const += v.get((co, d, h, e, q, k), 0)
    #                     if nonmaths_const > 0:
    #                         clash_expr += optional_maths * nonmaths_const
    # 
    # # ==================== GLOBAL UPPER BOUND CONSTRAINTS ====================
    # if soft_upper_bounds:
    #     print("Adding global upper bounds on soft constraints...")
    #     def add_global_bound(expr, bound, name):
    #         if bound is not None:
    #             # Skip zero expressions
    #             if isinstance(expr, (int, float)) and expr == 0:
    #                 return
    #             try:
    #                 p.addConstraint(expr <= bound, name)
    #             except Exception as e:
    #                 print(f"Error adding constraint {name}: {e}")
    #                 # Uncomment to see the expression (may be large)
    #                 # print(f"Expression: {expr}")
    #                 raise
    # 
    #     add_global_bound(late_expr, soft_upper_bounds.get('late'), "Global_Late_UB")
    #     add_global_bound(lunch_expr, soft_upper_bounds.get('lunch'), "Global_Lunch_UB")
    #     add_global_bound(wed_expr, soft_upper_bounds.get('wednesday'), "Global_Wed_UB")
    #     add_global_bound(isolated_expr, soft_upper_bounds.get('isolated'), "Global_Isolated_UB")
    #     add_global_bound(days_expr, soft_upper_bounds.get('days'), "Global_Days_UB")
    #     add_global_bound(clash_expr, soft_upper_bounds.get('clash'), "Global_Clash_UB")
    #     # Clash only for joint programmes
    # 
    # # ==================== OBJECTIVE ====================
    # p.setObjective(gwn, sense=xp.maximize)
    # print(f"Total constraints added: {constraint_count}")
    # 
    # var_dicts = {
    #     'xL': x_L, 'xW': x_W, 'xF': x_F,
    #     'yL': y_L, 'yW': y_W,
    #     'z': z, 'w': w_var,
    #     'a': a, 'ta': ta, 'lin': lin,
    #     'ind': ind, 'ch': ch,
    #     'is_isolated': is_isolated, 'b': b,
    #     'gwn': gwn
    # }
    # return p, constraint_list, var_dicts
    # ==================== PER‑CURRICULUM RAW SOFT EXPRESSIONS ====================
    # Compute raw expressions for late, lunch, wednesday, isolated, days, clash
    
    late_expr = {q:0 for q in QQ}
    lunch_expr = {q: 0 for q in QQ}
    wed_expr = {q: 0 for q in QQ}
    isolated_expr = {q: 0 for q in QQ}
    days_expr = {q: 0 for q in QQ}
    clash_expr = {q: 0 for q in Q}
    # ==================== PER‑CURRICULUM TRAVEL EXPRESSION ====================
    travel_expr = {q: 0 for q in Q}
    # Term 1: Maths → non‑maths (ind with co at h+1)
    for q in Q:
        for co in CO_q[q]:
            for g in G:
                for s in S:
                    weeks = E1 if s == 1 else E2
                    for e in weeks:
                        for d in D:
                            for h in H:
                                if h+1 not in H: continue
                                for k in K:
                                    val = v.get((co, d, h+1, e, q, k), 0)
                                    if val > 0:
                                        travel_expr[q] += travel_time_dict.get(('King\'s Buildings', k), 60) * val * ind[(q, co, g, d, h, s)]
    # Term 2: Non‑maths → Maths (ind with co at h)
    for q in Q:
        for co in CO_q[q]:
            for g in G:
                for s in S:
                    weeks = E1 if s == 1 else E2
                    for e in weeks:
                        for d in D:
                            for h in H:
                                if h+1 not in H: continue
                                for k in K:
                                    val = v.get((co, d, h, e, q, k), 0)
                                    if val > 0:
                                        travel_expr[q] += travel_time_dict.get((k, 'King\'s Buildings'), 60) * val * ind[(q, co, g, d, h+1, s)]
    # Term 3: Non‑maths ↔ non‑maths (using ch)
    for q in Q:
        for co in CO_q[q]:
            for co2 in CO_q[q]:
                if co < co2:
                    for s in S:
                        weeks = E1 if s == 1 else E2
                        for e in weeks:
                            for d in D:
                                for h in H:
                                    if h+1 not in H: continue
                                    for k in K:
                                        for k2 in K:
                                            val1 = v.get((co, d, h, e, q, k), 0)
                                            val2 = v.get((co2, d, h+1, e, q, k2), 0)
                                            if val1 > 0 and val2 > 0:
                                                travel_expr[q] += travel_time_dict.get((k, k2), 60) * val1 * val2 * ch[(q, co, co2)]
    for q in QQ:
        for s in S:
            weeks = E1 if s == 1 else E2
            for e in weeks:
                for d in D:
                    late_expr[q] += P[(q, d, late_hour, s, e)]
                    for h in lunch_hours:
                        lunch_expr[q] += P[(q, d, h, s, e)]
                    if d == wed_idx:
                        for h in wednesday_hours:
                            wed_expr[q] += P[(q, d, h, s, e)]
                for d in D:
                    for h in isolated_hours:
                        isolated_expr[q] += is_isolated[(q, d, h, s, e)]
                    days_expr[q] += b[(q, d, s, e)]

        # Clash expression (raw)
        if q != q_math:
            for s in S:
                weeks = E1 if s == 1 else E2
                for e in weeks:
                    for d in D:
                        for h in H:
                            optional_maths = xp.Sum(y_L[(o, d, h, s)] for o in O)
                            nonmaths = xp.Sum(v.get((co, d, h, e, q, k), 0) for co in CO_CO_q[q] for k in K)
                            clash_expr[q] += optional_maths * nonmaths

    # Add upper bound constraints
    for q in QQ:
        if 'late' in soft_upper_bounds and q in soft_upper_bounds['late']:
            p.addConstraint(late_expr[q] <= soft_upper_bounds['late'][q])
        if 'lunch' in soft_upper_bounds and q in soft_upper_bounds['lunch']:
            p.addConstraint(lunch_expr[q] <= soft_upper_bounds['lunch'][q])
        if 'wednesday' in soft_upper_bounds and q in soft_upper_bounds['wednesday']:
            p.addConstraint(wed_expr[q] <= soft_upper_bounds['wednesday'][q])
        if 'isolated' in soft_upper_bounds and q in soft_upper_bounds['isolated']:
            p.addConstraint(isolated_expr[q] <= soft_upper_bounds['isolated'][q])
        if 'days' in soft_upper_bounds and q in soft_upper_bounds['days']:
            p.addConstraint(days_expr[q] <= soft_upper_bounds['days'][q])
        if 'clash' in soft_upper_bounds and q in soft_upper_bounds['clash'] and q != 'math':
            p.addConstraint(clash_expr[q] <= soft_upper_bounds['clash'][q])
    # Per‑curriculum travel bounds
    if 'travel' in soft_upper_bounds:
        for q in Q:
            if q in soft_upper_bounds['travel']:
                p.addConstraint(travel_expr[q] <= soft_upper_bounds['travel'][q])
    # ==================== OBJECTIVE ====================
    # p.setBranchPriority(gwn,300)
    p.setObjective(gwn, sense=xp.maximize)

    print(f"Total constraints added: {constraint_count}")

    var_dicts = {
        'xL': x_L, 'xW': x_W, 'xF': x_F,
        'yL': y_L, 'yW': y_W,
        'z': z, 'w': w_var,
        'a': a, 'ta': ta, 'lin': lin,
        'ind': ind, 'ch': ch,
        'is_isolated': is_isolated, 'b': b,
        'gwn': gwn
    }
    return p, constraint_list, var_dicts

In [15]:
# Define global upper bounds (single number per soft term)
import math

# Updated raw bounds (from your latest extraction)
travel_upper = {
    'Economics and Mathematics (MA Hons): Year 3': 300.0,
    'Economics and Mathematics (MA Hons): Year 4': 600.0,
    'Philosophy and Mathematics (MA Hons): Year 3': 0.0,
    'Mathematics and Business BSc (Hons): Year 4': 300.0,
    'Computer Science and Mathematics (BSc Hons): Year 3': 930.0,
    'Computer Science and Mathematics (BSc Hons): Year 4': 0.0,
    'Mathematics and Physics (BSc Hons): Year 4': 0.0,
    'Mathematics and Business BSc (Hons): Year 3': 300.0,
    'Mathematics and Physics (BSc Hons): Year 3': 1200.0,
    'Philosophy and Mathematics (MA Hons): Year 4': 0.0,
    'math': 0.0
}

isolated_upper = {
    'Economics and Mathematics (MA Hons): Year 3': 1.0,
    'Economics and Mathematics (MA Hons): Year 4': 1.0,
    'Philosophy and Mathematics (MA Hons): Year 3': 0.0,
    'Mathematics and Business BSc (Hons): Year 4': 1.0,
    'Computer Science and Mathematics (BSc Hons): Year 3': 40.0,
    'Computer Science and Mathematics (BSc Hons): Year 4': 0.0,
    'Mathematics and Physics (BSc Hons): Year 4': 0.0,
    'Mathematics and Business BSc (Hons): Year 3': 1.0,
    'Mathematics and Physics (BSc Hons): Year 3': 21.0,
    'Philosophy and Mathematics (MA Hons): Year 4': 0.0,
    'math': 10.0
}

days_upper = {
    'Economics and Mathematics (MA Hons): Year 3': 50.0,
    'Economics and Mathematics (MA Hons): Year 4': 31.0,
    'Philosophy and Mathematics (MA Hons): Year 3': 31.0,
    'Mathematics and Business BSc (Hons): Year 4': 31.0,
    'Computer Science and Mathematics (BSc Hons): Year 3': 61.0,
    'Computer Science and Mathematics (BSc Hons): Year 4': 31.0,
    'Mathematics and Physics (BSc Hons): Year 4': 42.0,
    'Mathematics and Business BSc (Hons): Year 3': 31.0,
    'Mathematics and Physics (BSc Hons): Year 3': 53.0,
    'Philosophy and Mathematics (MA Hons): Year 4': 21.0,
    'math': 84.0
}

lunch_upper = {
    'Economics and Mathematics (MA Hons): Year 3': 49.0,
    'Economics and Mathematics (MA Hons): Year 4': 10.0,
    'Philosophy and Mathematics (MA Hons): Year 3': 0.0,
    'Mathematics and Business BSc (Hons): Year 4': 0.0,
    'Computer Science and Mathematics (BSc Hons): Year 3': 32.0,
    'Computer Science and Mathematics (BSc Hons): Year 4': 0.0,
    'Mathematics and Physics (BSc Hons): Year 4': 0.0,
    'Mathematics and Business BSc (Hons): Year 3': 0.0,
    'Mathematics and Physics (BSc Hons): Year 3': 0.0,
    'Philosophy and Mathematics (MA Hons): Year 4': 0.0,
    'math': 0.0
}

late_upper = {
    'Economics and Mathematics (MA Hons): Year 3': 0.0,
    'Economics and Mathematics (MA Hons): Year 4': 0.0,
    'Philosophy and Mathematics (MA Hons): Year 3': 0.0,
    'Mathematics and Business BSc (Hons): Year 4': 0.0,
    'Computer Science and Mathematics (BSc Hons): Year 3': 0.0,
    'Computer Science and Mathematics (BSc Hons): Year 4': 0.0,
    'Mathematics and Physics (BSc Hons): Year 4': 0.0,
    'Mathematics and Business BSc (Hons): Year 3': 0.0,
    'Mathematics and Physics (BSc Hons): Year 3': 0.0,
    'Philosophy and Mathematics (MA Hons): Year 4': 0.0,
    'math': 0.0
}

clash_upper = {
    'Economics and Mathematics (MA Hons): Year 3': 0,
    'Economics and Mathematics (MA Hons): Year 4': 10,
    'Philosophy and Mathematics (MA Hons): Year 3': 0,
    'Mathematics and Business BSc (Hons): Year 4': 10,
    'Computer Science and Mathematics (BSc Hons): Year 3': 12,
    'Computer Science and Mathematics (BSc Hons): Year 4': 0,
    'Mathematics and Physics (BSc Hons): Year 4': 0,
    'Mathematics and Business BSc (Hons): Year 3': 10,
    'Mathematics and Physics (BSc Hons): Year 3': 0,
    'Philosophy and Mathematics (MA Hons): Year 4': 0,
    'math': 0
}

wednesday_upper = {
    'Economics and Mathematics (MA Hons): Year 3': 0.0,
    'Economics and Mathematics (MA Hons): Year 4': 0.0,
    'Philosophy and Mathematics (MA Hons): Year 3': 0.0,
    'Mathematics and Business BSc (Hons): Year 4': 0.0,
    'Computer Science and Mathematics (BSc Hons): Year 3': 10.0,
    'Computer Science and Mathematics (BSc Hons): Year 4': 0.0,
    'Mathematics and Physics (BSc Hons): Year 4': 0.0,
    'Mathematics and Business BSc (Hons): Year 3': 0.0,
    'Mathematics and Physics (BSc Hons): Year 3': 0.0,
    'Philosophy and Mathematics (MA Hons): Year 4': 0.0,
    'math': 20.0
}

def bound_val(v):
    """Return ceil(1.15 * v), with a minimum of 1."""
    raw = math.ceil(1.15 * v)
    return max(1, raw)

# Build the final soft_upper_bounds dictionary
soft_upper_bounds = {}
for key, d in [
    ('travel', travel_upper),
    ('isolated', isolated_upper),
    ('days', days_upper),
    ('lunch', lunch_upper),
    ('late', late_upper),
    ('clash', clash_upper),
    ('wednesday', wednesday_upper)
]:
    soft_upper_bounds[key] = {q: bound_val(v) for q, v in d.items()}

# Optional: print examples
print("travel bound for Economics and Mathematics (MA Hons): Year 3:", soft_upper_bounds['travel']['Economics and Mathematics (MA Hons): Year 3'])
print("isolated bound for Computer Science and Mathematics (BSc Hons): Year 3:", soft_upper_bounds['isolated']['Computer Science and Mathematics (BSc Hons): Year 3'])
print("days bound for Mathematics and Physics (BSc Hons): Year 3:", soft_upper_bounds['days']['Mathematics and Physics (BSc Hons): Year 3'])

# Now pass soft_upper_bounds to the model building function
model, constraint_list, var_dicts = build_model_from_parameters(parameters, soft_upper_bounds)

travel bound for Economics and Mathematics (MA Hons): Year 3: 345
isolated bound for Computer Science and Mathematics (BSc Hons): Year 3: 46
days bound for Mathematics and Physics (BSc Hons): Year 3: 61
Creating decision variables...
Adding hard constraints...
QQ: ['Economics and Mathematics (MA Hons): Year 3', 'Economics and Mathematics (MA Hons): Year 4', 'Philosophy and Mathematics (MA Hons): Year 3', 'Mathematics and Business BSc (Hons): Year 4', 'Computer Science and Mathematics (BSc Hons): Year 3', 'Computer Science and Mathematics (BSc Hons): Year 4', 'Mathematics and Physics (BSc Hons): Year 4', 'Mathematics and Business BSc (Hons): Year 3', 'Mathematics and Physics (BSc Hons): Year 3', 'Philosophy and Mathematics (MA Hons): Year 4', 'math']
Total constraints added: 1012390


## Solve the model

The model is set to run for one hour, or until a solution gap of 10% is reached. The gap choice is arbitrary

In [16]:
import json
import os
import time
def save_solution_to_json(model, var_dicts, prefix="solution", var_to_val=None):
    """
    Save all variable groups to JSON files.
    If var_to_val is provided (a dict mapping var objects to values), it uses that;
    otherwise it retrieves values via model.getSolution() (slower).
    """
    if var_to_val is None:
        # Fallback to slow method (but we'll avoid this by always passing it)
        all_vars = []
        for var_group in var_dicts.values():
            if isinstance(var_group, dict):
                all_vars.extend(var_group.values())
            else:
                all_vars.append(var_group)
        sol_vals = model.getSolution(all_vars)
        var_to_val = {v: val for v, val in zip(all_vars, sol_vals)}
    
    for name, var_group in var_dicts.items():
        # Skip entries that are not dictionaries (e.g., single variables)
        if not isinstance(var_group, dict):
            print(f"   Skipping {name} (not a dict)")
            continue
        data = {}
        for key, var in var_group.items():
            data[str(key)] = var_to_val[var]
        filename = f"{prefix}_{name}.json"
        with open(filename, "w") as f:
            json.dump(data, f)
        print(f"   Saved {name} → {filename} ({len(data):,} entries)")
    print(f"\nAll solutions saved with prefix '{prefix}'")

def load_var_sol_from_json(prefix="solution"):
    """
    Load the saved solution dictionaries back into memory.
    """
    var_sol = {}
    import glob
    for file in glob.glob(f"{prefix}_*.json"):
        group = file[len(prefix)+1:-5]
        with open(file, "r") as f:
            data_str = json.load(f)
        data = {}
        for k_str, val in data_str.items():
            import ast
            key = ast.literal_eval(k_str)
            data[key] = val
        var_sol[group] = data
        print(f"Loaded {group} ({len(data):,} entries)")
    return var_sol

In [17]:
def solve_model(parameters, soft_upper_bounds=None):
    from xpress import InterfaceError
    import time

    if soft_upper_bounds is None:
        soft_upper_bounds = {}
    model, constraint_list, var_dicts = build_model_from_parameters(parameters, soft_upper_bounds)
    row_to_name = {r: name for r, name in constraint_list}
    # model.setControl('HEURFREQ', 0)
    model.setControl('outputlog', 1)
    # model.setControl('miprelstop', 0.)
    # model.setControl('maxtime', 3600)
    model.solve()
    
    status_string = model.getProbStatusString()
    print("\nSolution status:", status_string)
    
    if "optimal" in status_string or "Feasible" in status_string:
        print("Feasible solution found!")
        
        # --- Fast retrieval of all variable values ---
        print("Retrieving solution values...")
        start = time.time()
        
        # Collect all variable objects (handling both dicts and single vars)
        all_vars = []
        for var_group in var_dicts.values():
            if isinstance(var_group, dict):
                all_vars.extend(var_group.values())
            else:
                all_vars.append(var_group)   # single variable (e.g., gwn)
        
        # Get all values in one call
        sol_vals = model.getSolution(all_vars)
        var_to_val = {v: val for v, val in zip(all_vars, sol_vals)}
        
        print(f"Retrieved {len(all_vars)} values in {time.time()-start:.2f}s")
        
        # Save to JSON (keeps a permanent copy)
        save_solution_to_json(model, var_dicts, prefix="solution", var_to_val=var_to_val)
        
        # Build var_sol for needed groups (used for results)
        needed = ['z', 'w', 'a', 'ta', 'lin', 'xL', 'xW', 'xF', 'yL', 'yW',
                  'ind', 'ch', 'is_isolated', 'b']
        var_sol = {}
        for name in needed:
            if name in var_dicts and isinstance(var_dicts[name], dict):
                var_sol[name] = {key: var_to_val[var] for key, var in var_dicts[name].items()}
        
        results = analyze_solution(model, parameters, model.getSolution(), var_sol)
        return model, results, var_sol
    else:
        print("Model is infeasible!")
        print("\nComputing IIS...")
        
        model.iisfirst(1)
        rows = None
        bounds = None
        try:
            rows = model.attributes.iisrows
            bounds = model.attributes.iisbnds
        except (InterfaceError, AttributeError):
            pass
        
        if rows is None:
            try:
                rows = model.getiisrows()
                bounds = model.getiisbnds()
            except AttributeError:
                pass
        
        if rows is None:
            model.write("iis.ilp")
            print("IIS written to 'iis.ilp'. Please open this file to inspect the constraints.")
            return model, None, None
        
        print(f"\nFound {len(rows)} constraints in the IIS.")
        if rows:
            print("Constraints in IIS:")
            for row_idx in rows:
                if row_idx in row_to_name:
                    print(f"  - {row_to_name[row_idx]} (row {row_idx})")
                else:
                    print(f"  - Row {row_idx} (no name stored)")
        
        print(f"\nFound {len(bounds)} variable bounds in the IIS.")
        if bounds:
            print("Variable bounds in IIS:")
            for col_idx, lower, upper in bounds:
                var_name = model.getVarName(col_idx)
                print(f"  - {var_name}: lower={lower}, upper={upper}")
        
        model.write("iis.ilp")
        print("\nIIS written to 'iis.ilp'. You can open this file to see the exact constraints.")
        return model, None, None
def analyze_solution(model, parameters, sol, var_sol):
    """
    Analyze and display the solution using the variable solution dictionaries.
    """
    results = {
        'gateway_schedule': {},      # gateway -> semester(s)
        'optional_schedule': {},     # optional -> semester(s)
        'course_selection': {},      # curriculum -> list of chosen non‑maths courses
        'gateway_choice': {}         # curriculum -> list of (gateway, semester)
    }
    
    # 1. Gateway semester assignment (z)
    for (g, s), val in var_sol['z'].items():
        if val > 0.5:
            if g not in results['gateway_schedule']:
                results['gateway_schedule'][g] = []
            results['gateway_schedule'][g].append(s)
    
    # 2. Optional course semester assignment (w_var)
    for (o, s), val in var_sol['w'].items():
        if val > 0.5:
            if o not in results['optional_schedule']:
                results['optional_schedule'][o] = []
            results['optional_schedule'][o].append(s)
    
    # 3. Non‑maths course selection per curriculum (a)
    for (q, co), val in var_sol['a'].items():
        if val > 0.5:
            if q not in results['course_selection']:
                results['course_selection'][q] = []
            results['course_selection'][q].append(co)
    
    # 4. Gateway pathway choice per curriculum (ta)
    for (g, s, q), val in var_sol['ta'].items():
        if val > 0.5:
            if q not in results['gateway_choice']:
                results['gateway_choice'][q] = []
            results['gateway_choice'][q].append((g, s))
    
    return results
def print_results(results):
    """
    Print the results from analyze_solution.
    """
    if results is None:
        print("No results to display")
        return
    
    print("\n" + "="*50)
    print("SOLUTION RESULTS")
    print("="*50)
    
    print("\nGateway Course Schedule:")
    for g, sems in results['gateway_schedule'].items():
        print(f"  {g}: Semester(s) {sems}")
    
    print("\nOptional Course Schedule:")
    for o, sems in results['optional_schedule'].items():
        print(f"  {o}: Semester(s) {sems}")
    
    print("\nNon‑Maths Course Selections per Curriculum:")
    for q, courses in results['course_selection'].items():
        print(f"  {q}: {courses}")
    
    print("\nGateway Pathway Choices per Curriculum (gateway, semester):")
    for q, choices in results['gateway_choice'].items():
        print(f"  {q}: {choices}")
    
    print("\nFeasibility Status:")
    print("  Model is feasible - all constraints satisfied.")

In [ ]:
# Assuming you have already run your extraction steps
# parameters = extract_all_parameters(courses_df, programme_df)

# Build and solve the model
model, results, var_sol = solve_model(parameters, soft_upper_bounds)

# Print results
print_results(results)

Creating decision variables...
Adding hard constraints...
QQ: ['Economics and Mathematics (MA Hons): Year 3', 'Economics and Mathematics (MA Hons): Year 4', 'Philosophy and Mathematics (MA Hons): Year 3', 'Mathematics and Business BSc (Hons): Year 4', 'Computer Science and Mathematics (BSc Hons): Year 3', 'Computer Science and Mathematics (BSc Hons): Year 4', 'Mathematics and Physics (BSc Hons): Year 4', 'Mathematics and Business BSc (Hons): Year 3', 'Mathematics and Physics (BSc Hons): Year 3', 'Philosophy and Mathematics (MA Hons): Year 4', 'math']
Total constraints added: 1012390
FICO Xpress v9.7.0, Hyper, solve started 20:34:07, Apr 3, 2026
Heap usage: 402MB (peak 402MB, 43MB system)
Minimizing MILP noname using up to 8 threads and up to 7975MB memory, with these control settings:
OUTPUTLOG = 1
NLPPOSTSOLVE = 1
XSLP_DELETIONCONTROL = 0
XSLP_OBJSENSE = 1
Original problem has:
   1012506 rows       262378 cols      3035502 elements    262378 entities
Presolved problem has:
    273278

In [ ]:
def extract_per_curriculum_components(parameters, var_sol):
    """
    Extracts raw counts and scaled contributions for each curriculum.
    Returns dictionaries: travel_raw, clash_raw, late_raw, lunch_raw, wed_raw, isolated_raw, days_raw,
                         and the scaled versions (with the same scaling as in the model).
    """
    G = parameters['G']
    O = parameters['O']
    S = parameters['S']
    D = parameters['D']
    H = parameters['H']
    Q = parameters['Q']
    E1 = parameters['E1']
    E2 = parameters['E2']
    K = parameters['K']
    v = parameters['v']
    programme_data = parameters['programme_data']
    CO_q = {prog: programme_data[prog]['all_courses'] for prog in Q}

    # Solution dictionaries
    xL = var_sol.get('xL', {})
    xW = var_sol.get('xW', {})
    xF = var_sol.get('xF', {})
    yL = var_sol.get('yL', {})
    yW = var_sol.get('yW', {})
    a  = var_sol.get('a', {})
    ind = var_sol.get('ind', {})
    ch  = var_sol.get('ch', {})
    is_isolated_sol = var_sol.get('is_isolated', {})
    b_sol = var_sol.get('b', {})
    lin = var_sol.get('lin', {})

    q_math = 'math'
    QQ = list(Q) + [q_math]

    # Initialise dictionaries
    travel_raw_per_q = {q: 0.0 for q in Q}
    clash_raw_per_q = {q: 0 for q in Q}
    late_raw_per_q = {q: 0 for q in QQ}
    lunch_raw_per_q = {q: 0 for q in QQ}
    wed_raw_per_q = {q: 0 for q in QQ}
    isolated_raw_per_q = {q: 0 for q in QQ}
    days_raw_per_q = {q: 0 for q in QQ}

    # ------------------------------------------------------------------
    # 1. Travel raw (minutes) – unchanged
    # ------------------------------------------------------------------
    for q in Q:
        for co in CO_q[q]:
            if co in G or co in O: continue
            for g in G:
                for s in S:
                    weeks = E1 if s == 1 else E2
                    for e in weeks:
                        for d in D:
                            for h in H:
                                if h + 1 not in H: continue
                                # Gateway at h → non-maths at h+1
                                ind_val = ind.get((q, co, g, d, h, s), 0)
                                if ind_val > 0.5:
                                    for k in K:
                                        if v.get((co, d, h+1, e, q, k), 0) > 0:
                                            travel_raw_per_q[q] += travel_time_dict.get(("King's Buildings", k), 60)

                                # non-maths at h → Gateway at h+1
                                ind_val2 = ind.get((q, co, g, d, h+1, s), 0)
                                if ind_val2 > 0.5:
                                    for k in K:
                                        if v.get((co, d, h, e, q, k), 0) > 0:
                                            travel_raw_per_q[q] += travel_time_dict.get((k, "King's Buildings"), 60)

    # non-maths <-> non-maths
    for q in Q:
        nonmath = [c for c in CO_q[q] if c not in G and c not in O]
        for i, co1 in enumerate(nonmath):
            for co2 in nonmath[i+1:]:
                ch_val = ch.get((q, co1, co2), 0)
                if ch_val <= 0.5: continue
                for s in S:
                    weeks = E1 if s == 1 else E2
                    for e in weeks:
                        for d in D:
                            for h in H:
                                if h + 1 not in H: continue
                                for k1 in K:
                                    for k2 in K:
                                        if (v.get((co1,d,h,e,q,k1),0) > 0 and 
                                            v.get((co2,d,h+1,e,q,k2),0) > 0):
                                            travel_raw_per_q[q] += travel_time_dict.get((k1, k2), 60) * ch_val

    # ------------------------------------------------------------------
    # 2. Clash raw – unchanged
    # ------------------------------------------------------------------
    for q in Q:
        for s in S:
            weeks = E1 if s == 1 else E2
            for e in weeks:
                for d in D:
                    for h in H:
                        opt_math = any(yL.get((o,d,h,s),0) > 0.5 for o in O) 
                        if not opt_math: continue
                        non_count = 0
                        for co in CO_q[q]:
                            if co not in G and co not in O:
                                if a.get((q,co),0) > 0.5:
                                    for k in K:
                                        if v.get((co,d,h,e,q,k),0) > 0:
                                            non_count += 1
                                            break
                        clash_raw_per_q[q] += non_count

    # ------------------------------------------------------------------
    # 3. Late / Lunch / Wednesday / Isolated / Days (per curriculum)
    # ------------------------------------------------------------------
    for q in QQ:
        for s in S:
            weeks = E1 if s == 1 else E2
            for e in weeks:
                for d in D:
                    # Event presence for each hour
                    for h in H:
                        if q == q_math:
                            p = 0
                            # Gateway lectures
                            p += sum(xL.get((g, d, h, s), 0) for g in G)
                            # Gateway weekly workshops
                            # Gateway fortnightly workshops (depending on week parity)
    
                            # Optional maths lectures
                            p += sum(yL.get((o, d, h, s), 0) for o in O)
                            # Optional maths weekly workshops
                        else:
                            # Joint curriculum: only gateway lectures that are taken
                            p = sum(lin.get((g, d, h, s, q), 0) for g in G)
                            # Non‑maths courses
                            for co in CO_q[q]:
                                if co in G or co in O: continue
                                if a.get((q, co), 0) > 0.5:
                                    p += sum(v.get((co, d, h, e, q, k), 0) for k in K)

                        if h == 17:
                            late_raw_per_q[q] += p
                        if h in (12, 13):
                            lunch_raw_per_q[q] += p
                        if d == 3 and 13 <= h <= 17:
                            wed_raw_per_q[q] += p

                        # Isolated classes
                        if 10 <= h <= 16:
                            isolated_raw_per_q[q] += is_isolated_sol.get((q, d, h, s, e), 0)

                    # Days used (once per day)
                    days_raw_per_q[q] += b_sol.get((q, d, s, e), 0)

    # ------------------------------------------------------------------
    # Scaling (same as in the model)
    # ------------------------------------------------------------------
    weeks_per_sem = len(E1)
    N_late   = len(S) * len(D) * weeks_per_sem
    N_lunch  = len(S) * len(D) * 2 * weeks_per_sem
    N_isol   = len(S) * len(D) * 7 * weeks_per_sem
    N_days   = len(S) * len(D) * weeks_per_sem
    N_wed    = len(S) * 5 * weeks_per_sem
    N_clash  = len(S) * weeks_per_sem * 3 * 2
    max_travel_sem = 660 * weeks_per_sem * len(S)

    travel_scaled_per_q = {q: 0.01 * travel_raw_per_q[q] / max_travel_sem for q in Q}
    clash_scaled_per_q = {q: 0.1 * clash_raw_per_q[q] / N_clash for q in Q}
    late_scaled_per_q = {q: 0.344249 * late_raw_per_q[q] / N_late for q in QQ}
    lunch_scaled_per_q = {q: 0.112523 * lunch_raw_per_q[q] / N_lunch for q in QQ}
    wed_scaled_per_q = {q: 0.028978 * wed_raw_per_q[q] / N_wed for q in QQ}
    isolated_scaled_per_q = {q: 0.057430 * isolated_raw_per_q[q] / N_isol for q in QQ}
    days_scaled_per_q = {q: 0.133207 * days_raw_per_q[q] / N_days for q in QQ}

    return {
        'travel_raw': travel_raw_per_q,
        'clash_raw': clash_raw_per_q,
        'late_raw': late_raw_per_q,
        'lunch_raw': lunch_raw_per_q,
        'wed_raw': wed_raw_per_q,
        'isolated_raw': isolated_raw_per_q,
        'days_raw': days_raw_per_q,
        'travel_scaled': travel_scaled_per_q,
        'clash_scaled': clash_scaled_per_q,
        'late_scaled': late_scaled_per_q,
        'lunch_scaled': lunch_scaled_per_q,
        'wed_scaled': wed_scaled_per_q,
        'isolated_scaled': isolated_scaled_per_q,
        'days_scaled': days_scaled_per_q,
    }

In [ ]:
# After solving and printing results
per_q = extract_per_curriculum_components(parameters, var_sol)

print("\n" + "="*80)
print("PER‑CURRICULUM SOFT CONSTRAINT VALUES (RAW)")
print("="*80)

# Print header
print(f"{'Curriculum':<50} {'Travel':>8} {'Clash':>6} {'Late':>5} {'Lunch':>6} {'Wed':>5} {'Isolated':>9} {'Days':>5}")
print("-"*80)

# Print each curriculum (including 'math')
for q in per_q['travel_raw']:
    travel = per_q['travel_raw'][q]
    clash = per_q['clash_raw'][q]
    late = per_q['late_raw'][q]
    lunch = per_q['lunch_raw'][q]
    wed = per_q['wed_raw'][q]
    isolated = per_q['isolated_raw'][q]
    days = per_q['days_raw'][q]
    print(f"{q:<50} {travel:8.0f} {clash:6d} {late:5.0f} {lunch:6.0f} {wed:5.0f} {isolated:9.0f} {days:5.0f}")

print("\n" + "="*80)
print("PER‑CURRICULUM SCALED CONTRIBUTIONS")
print("="*80)
print(f"{'Curriculum':<50} {'Travel':>8} {'Clash':>6} {'Late':>6} {'Lunch':>6} {'Wed':>6} {'Isolated':>8} {'Days':>6}")
print("-"*80)
for q in per_q['travel_scaled']:
    travel_sc = per_q['travel_scaled'][q]
    clash_sc = per_q['clash_scaled'][q]
    late_sc = per_q['late_scaled'][q]
    lunch_sc = per_q['lunch_scaled'][q]
    wed_sc = per_q['wed_scaled'][q]
    isolated_sc = per_q['isolated_scaled'][q]
    days_sc = per_q['days_scaled'][q]
    print(f"{q:<50} {travel_sc:8.4f} {clash_sc:6.4f} {late_sc:6.4f} {lunch_sc:6.4f} {wed_sc:6.4f} {isolated_sc:8.4f} {days_sc:6.4f}")

In [ ]:
print("Curricula in Q:", parameters['Q'])

## Visualise results

In [ ]:
def plot_curriculum_timetable(q, semester, var_sol, parameters,
                               week=None,
                               day_names=['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
                               hour_range=(9, 17),
                               figsize=(12, 6)):
    """
    Plot timetable for curriculum q using solution dictionaries.
    """
    import numpy as np
    import matplotlib.pyplot as plt
    
    # Unpack needed data
    G = parameters['G']
    # Get all courses for this curriculum from programme_data
    if q not in parameters['programme_data']:
        print(f"Curriculum '{q}' not found in programme_data.")
        return
    CO_q = parameters['programme_data'][q]['all_courses']
    v = parameters['v']
    D = parameters['D']          # day numbers (e.g., 1..5)
    H = parameters['H']          # hour numbers (e.g., 9..18)
    E1 = parameters['E1']
    E2 = parameters['E2']
    K = parameters['K']
    
    # Get solution dictionaries
    lin_sol = var_sol['lin']
    a_sol = var_sol['a']
    print('lin sol:',lin_sol)
    # Set weeks
    weeks = E1 if semester == 1 else E2
    
    # Map day numbers to indices
    day_map = {d: i for i, d in enumerate(day_names)}
    day_index_to_name = {i+1: day_names[i] for i in range(len(day_names))}
    
    # Hours
    hours = list(range(hour_range[0], hour_range[1]+1))
    n_hours = len(hours)
    n_days = len(day_names)
    
    # Occupancy grid
    if week is None:
        occupancy = np.zeros((n_days, n_hours))
    else:
        occupancy = np.zeros((n_days, n_hours), dtype=bool)
        labels = [[[] for _ in hours] for _ in day_names]
    
    # Loop over days, hours, weeks
    for d in D:
        day_name = day_index_to_name.get(d)
        if day_name is None:
            continue
        day_idx = day_map[day_name]
        for h in hours:
            if h not in H:
                continue
            hour_idx = h - hour_range[0]
            if hour_idx < 0 or hour_idx >= n_hours:
                continue
            for e in weeks:
                # Gateway events: lin = 1
                gateway = False
                for g in G:
                    # lin key: (g, d, h, semester, q)
                    if lin_sol.get((g, d, h, semester, q), 0) > 0.5:
                        gateway = True
                        break
                # Non‑maths events: a=1 and v=1 for some k
                nonmaths = False
                for co in CO_q:
                    if a_sol.get((q, co), 0) > 0.5:
                        for k in K:
                            if v.get((co, d, h, e, q, k), 0) > 0:
                                nonmaths = True
                                break
                    if nonmaths:
                        break
                if gateway or nonmaths:
                    if week is None:
                        occupancy[day_idx, hour_idx] += 1
                    else:
                        if e == week:
                            occupancy[day_idx, hour_idx] = True
                            label_parts = []
                            if gateway:
                                label_parts.append("Gateway")
                            if nonmaths:
                                label_parts.append("Non-Maths")
                            labels[day_idx][hour_idx].append("\n".join(label_parts))
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    if week is None:
        im = ax.imshow(occupancy, cmap='YlOrRd', aspect='auto',
                       vmin=0, vmax=occupancy.max() if occupancy.max() > 0 else 1)
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Number of weeks occupied')
    else:
        im = ax.imshow(occupancy, cmap='Blues', aspect='auto', vmin=0, vmax=1)
        for i in range(n_days):
            for j in range(n_hours):
                if occupancy[i, j]:
                    text = '\n'.join(labels[i][j])
                    ax.text(j, i, text, ha='center', va='center',
                            fontsize=8, color='black', weight='bold')
    
    ax.set_xticks(range(n_hours))
    ax.set_xticklabels([f'{h}:00' for h in hours])
    ax.set_yticks(range(n_days))
    ax.set_yticklabels(day_names)
    ax.set_xlabel('Hour')
    ax.set_ylabel('Day')
    title = f"Timetable for {q} – Semester {semester}"
    if week:
        title += f", Week {week}"
    else:
        title += " (all weeks combined)"
    ax.set_title(title)
    ax.set_xticks(np.arange(-0.5, n_hours, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_days, 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
    plt.tight_layout()
    plt.show()

In [ ]:
programmes_needed = ['Mathematics and Business BSc (Hons): Year 3',
                     # 'Mathematics and Physics (BSc Hons)',
                     'Economics and Mathematics (MA Hons): Year 3','Computer Science and Mathematics (BSc Hons): Year 3',
              'Philosophy and Mathematics (MA Hons): Year 3']

for prog in programmes_needed:
    plot_curriculum_timetable(prog, semester = 1, var_sol = var_sol, parameters = parameters, week = 9)

In [ ]:
# After solving and retrieving the solution dictionaries
# For example, for the 'Economics and Mathematics (MA Hons)' curriculum in Semester 1:
# 
plot_curriculum_timetable('Economics and Mathematics (MA Hons): Year 3', semester=2, 
                          var_sol=var_sol, parameters=parameters, week = 26)# 

# For all weeks combined:
#plot_curriculum_timetable('Economics and Mathematics (MA Hons)', semester=1, week=None)

In [ ]:
plot_curriculum_timetable('Economics and Mathematics (MA Hons): Year 3', semester=1,  var_sol = var_sol, parameters = parameters,week=None)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_physics_timetable(courses_df, semester=1, week=None, 
                           day_names=['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
                           hour_range=(9, 17),  # 9am to 5pm
                           figsize=(12, 6)):
    """
    Plot the timetable of physics courses.
    
    Parameters:
    - courses_df: DataFrame containing course event data (must have columns:
                  'Module Name', 'Module Department', 'Semester', 'Timeslot',
                  'Duration (minutes)', 'Weeks', 'Campus', etc.)
    - semester: 1 or 2 (default 1)
    - week: int (1-52) or None. If None, shows all weeks (color intensity = number of weeks the slot is occupied).
    - day_names: list of day labels (default Mon-Fri)
    - hour_range: tuple (start_hour, end_hour) in 24h format (default 9-17)
    - figsize: figure size
    """
    # Filter for physics courses
    physics_df = courses_df[courses_df['Module Department'] == 'School of Physics and Astronomy'].copy()
    
    if physics_df.empty:
        print("No physics courses found.")
        return
    
    # Parse timeslot and weeks
    def parse_timeslot(timeslot):
        if pd.isna(timeslot):
            return None, None
        parts = str(timeslot).split()
        if len(parts) >= 2:
            day = parts[0]
            time_str = parts[1]
            hour = int(time_str.split(':')[0])
            return day, hour
        return None, None
    
    def parse_weeks(weeks_str):
        if pd.isna(weeks_str):
            return []
        weeks = []
        for part in str(weeks_str).split(','):
            if '-' in part:
                start, end = map(int, part.split('-'))
                weeks.extend(range(start, end + 1))
            else:
                weeks.append(int(part))
        return weeks
    
    # Map days to indices
    day_map = {'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':3, 'Friday':4,
               'Saturday':5, 'Sunday':6}
    
    # Create grid for hours
    hours = list(range(hour_range[0], hour_range[1]+1))
    n_hours = len(hours)
    n_days = len(day_names)
    
    # Initialize occupancy grid (days x hours)
    if week is None:
        # For all weeks, store count of weeks occupied
        occupancy = np.zeros((n_days, n_hours))
    else:
        # For a specific week, store binary occupancy
        occupancy = np.zeros((n_days, n_hours), dtype=bool)
    
    # Collect events to display labels (for specific week only)
    labels = [[[] for _ in hours] for _ in day_names]  # only if week is specific
    
    # For each physics course
    for _, row in physics_df.iterrows():
        # Check semester
        sem = row.get('Semester')
        if isinstance(sem, str):
            if sem == 'Semester 1':
                sem_num = 1
            elif sem == 'Semester 2':
                sem_num = 2
            else:
                continue
        else:
            sem_num = sem
        if sem_num != semester:
            continue
        
        # Parse timeslot
        day_str, hour = parse_timeslot(row['Timeslot'])
        if day_str is None or hour is None:
            continue
        if day_str not in day_map:
            continue
        day_idx = day_map[day_str]
        # Check hour within range
        if hour < hour_range[0] or hour > hour_range[1]:
            continue
        hour_idx = hour - hour_range[0]
        
        # Get weeks
        weeks = parse_weeks(row['Weeks'])
        if not weeks:
            continue
        # If week is specified, only consider that week
        if week is not None:
            if week not in weeks:
                continue
            occupancy[day_idx, hour_idx] = True
            # Add course name to labels
            labels[day_idx][hour_idx].append(row['Module Name'])
        else:
            # Count how many weeks this event covers
            occupancy[day_idx, hour_idx] += len(weeks)
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    if week is None:
        # Show occupancy as heatmap (number of weeks)
        im = ax.imshow(occupancy, cmap='YlOrRd', aspect='auto', 
                       vmin=0, vmax=occupancy.max() if occupancy.max() > 0 else 1)
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Number of weeks occupied')
    else:
        # Show as binary occupied cells
        im = ax.imshow(occupancy, cmap='Blues', aspect='auto', vmin=0, vmax=1)
        # Add text labels
        for i in range(n_days):
            for j in range(n_hours):
                if occupancy[i, j]:
                    # Combine course names (if multiple)
                    courses_text = '\n'.join(labels[i][j])
                    ax.text(j, i, courses_text, ha='center', va='center',
                            fontsize=8, color='black', weight='bold')
    
    # Set axes
    ax.set_xticks(range(n_hours))
    ax.set_xticklabels([f'{h}:00' for h in hours])
    ax.set_yticks(range(n_days))
    ax.set_yticklabels(day_names)
    ax.set_xlabel('Hour')
    ax.set_ylabel('Day')
    title = f"Physics Courses Timetable – Semester {semester}"
    if week:
        title += f", Week {week}"
    else:
        title += " (all weeks combined)"
    ax.set_title(title)
    
    # Grid lines
    ax.set_xticks(np.arange(-0.5, n_hours, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_days, 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot all physics courses in Semester 1 (all weeks combined)
plot_physics_timetable(courses_df, semester=1, week=None)

# Plot physics courses for Week 10 in Semester 1
plot_physics_timetable(courses_df, semester=1, week=10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def plot_compulsory_timetable(programme_name, programme_data, courses_df, 
                              semester=1, week=None,
                              day_names=['Mon', 'Tue', 'Wed', 'Thu', 'Fri'],
                              hour_range=(9, 17),   # 9am to 5pm
                              figsize=(12, 6)):
    """
    Plot the timetable of compulsory courses for a given programme.
    
    Parameters:
    - programme_name: string (e.g., 'Mathematics and Physics (BSc Hons)')
    - programme_data: dictionary returned by extract_programme_structure()
    - courses_df: DataFrame containing course event data
    - semester: 1 or 2
    - week: int (1-52) or None. If None, shows all weeks (heatmap of occupancy count)
    - day_names: list of day labels
    - hour_range: tuple (start_hour, end_hour) in 24h format
    - figsize: figure size
    """
    # Get compulsory courses for the programme
    if programme_name not in programme_data:
        print(f"Programme '{programme_name}' not found in programme data.")
        return
    
    compulsory_courses = programme_data[programme_name]['compulsory_courses']
    if not compulsory_courses:
        print(f"No compulsory courses found for {programme_name}")
        return
    
    print(f"Compulsory courses for {programme_name}: {compulsory_courses}")
    
    # Filter timetable data for these courses
    relevant_df = courses_df[courses_df['Module Name'].isin(compulsory_courses)].copy()
    if relevant_df.empty:
        print("No timetable data found for these courses.")
        return
    
    # Parse timeslot and weeks
    def parse_timeslot(timeslot):
        if pd.isna(timeslot):
            return None, None
        parts = str(timeslot).split()
        if len(parts) >= 2:
            day = parts[0]
            time_str = parts[1]
            hour = int(time_str.split(':')[0])
            return day, hour
        return None, None
    
    def parse_weeks(weeks_str):
        if pd.isna(weeks_str):
            return []
        weeks = []
        for part in str(weeks_str).split(','):
            if '-' in part:
                start, end = map(int, part.split('-'))
                weeks.extend(range(start, end + 1))
            else:
                weeks.append(int(part))
        return weeks
    
    # Map days to indices
    day_map = {'Monday':0, 'Tuesday':1, 'Wednesday':2, 'Thursday':3, 'Friday':4,
               'Saturday':5, 'Sunday':6}
    
    # Create grid
    hours = list(range(hour_range[0], hour_range[1]+1))
    n_hours = len(hours)
    n_days = len(day_names)
    
    # Initialize occupancy grid
    if week is None:
        occupancy = np.zeros((n_days, n_hours))   # count of weeks
    else:
        occupancy = np.zeros((n_days, n_hours), dtype=bool)
        labels = [[[] for _ in hours] for _ in day_names]
    
    # Process each row
    for _, row in relevant_df.iterrows():
        # Check semester
        sem = row.get('Semester')
        if isinstance(sem, str):
            if sem == 'Semester 1':
                sem_num = 1
            elif sem == 'Semester 2':
                sem_num = 2
            else:
                continue
        else:
            sem_num = sem
        if sem_num != semester:
            continue
        
        # Parse timeslot
        day_str, hour = parse_timeslot(row['Timeslot'])
        if day_str is None or hour is None:
            continue
        if day_str not in day_map:
            continue
        day_idx = day_map[day_str]
        if day_idx >= n_days:  # only show requested days
            continue
        if hour < hour_range[0] or hour > hour_range[1]:
            continue
        hour_idx = hour - hour_range[0]
        
        # Get weeks
        weeks = parse_weeks(row['Weeks'])
        if not weeks:
            continue
        
        if week is not None:
            if week in weeks:
                occupancy[day_idx, hour_idx] = True
                # Add course name to labels (maybe include course code as well)
                course_name = row['Module Name']
                labels[day_idx][hour_idx].append(course_name)
        else:
            occupancy[day_idx, hour_idx] += len(weeks)
    
    # Plot
    fig, ax = plt.subplots(figsize=figsize)
    
    if week is None:
        im = ax.imshow(occupancy, cmap='YlOrRd', aspect='auto',
                       vmin=0, vmax=occupancy.max() if occupancy.max() > 0 else 1)
        cbar = plt.colorbar(im, ax=ax)
        cbar.set_label('Number of weeks occupied')
    else:
        im = ax.imshow(occupancy, cmap='Blues', aspect='auto', vmin=0, vmax=1)
        # Add text labels
        for i in range(n_days):
            for j in range(n_hours):
                if occupancy[i, j]:
                    # Combine course names
                    courses_text = '\n'.join(labels[i][j])
                    ax.text(j, i, courses_text, ha='center', va='center',
                            fontsize=8, color='black', weight='bold')
    
    # Set axes
    ax.set_xticks(range(n_hours))
    ax.set_xticklabels([f'{h}:00' for h in hours])
    ax.set_yticks(range(n_days))
    ax.set_yticklabels(day_names[:n_days])
    ax.set_xlabel('Hour')
    ax.set_ylabel('Day')
    title = f"Compulsory Courses – {programme_name} (Semester {semester})"
    if week:
        title += f", Week {week}"
    else:
        title += " (all weeks combined)"
    ax.set_title(title)
    
    # Grid lines
    ax.set_xticks(np.arange(-0.5, n_hours, 1), minor=True)
    ax.set_yticks(np.arange(-0.5, n_days, 1), minor=True)
    ax.grid(which='minor', color='gray', linestyle='-', linewidth=0.5)
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Plot all weeks for Semester 1
plot_compulsory_timetable('Mathematics and Physics (BSc Hons)', 
                          programme_data, courses_df, 
                          semester=1, week=None)

# Plot a specific week (e.g., week 10)
plot_compulsory_timetable('Mathematics and Physics (BSc Hons)', 
                          programme_data, courses_df, 
                          semester=1, week=10)